# Validate the analytics views

## Purpose

Check the six Analytics views before any number reaches the dashboard. The contract is [D-56](../../docs/decisions.md) (proposed): exactly one view per capstone insight.

| # | View | One row per |
| --- | --- | --- |
| 1 | `analytics_investment_distribution` | Geography level (region or province reporting area), geography key, and `infra_year` cohort |
| 2 | `analytics_population_equity` | Province reporting area (D-57) |
| 3 | `analytics_flood_alignment` | Official region and flood point class |
| 4 | `analytics_implementation_monitoring` | Current DPWH project |
| 5 | `analytics_infrastructure_network` | Current DPWH project |
| 6 | `analytics_integration_coverage` | Integration link, link outcome, and region key |

The notebook first proves that `05-analytics` holds exactly these six views and none of the retired ones. Then it rebuilds each view straight from Gold with its own SQL and compares it row by row. The expected values live here, not in the analytics notebook, so a view never checks itself. The notebook only reads Gold, the views, and the validation results. It writes only `04-validation.analytics_dq_results`, and it saves its evidence **before** the STOP gate runs, so a failed check stays visible. The table has the same columns as `gold_dq_results`, with `layer = 'analytics'`.

## Inputs

- `03-gold` facts and dimensions, including the D-57 province reporting-area dimension, boundary dimension, and population fact, to rebuild every view on its own
- The six analytics views in `05-analytics`, and `information_schema.views` to list every view in that schema
- `04-validation.gold_dq_results`, the Gold STOP gates for the regional model and the D-57 province model
- `04-validation.analytics_dq_results`, the saved evidence of an earlier run on the same Gold data, for the rerun check

## Cost

Gold is read once into one current-project view that every rebuild reuses. The largest input is the project fact. Two views have one row per project, so their comparison is a join on `project_key`. Every check comes from one metrics row per view, and the gate reads the saved results.

## 1. Set the expected values and read the runs

**What:** set the expected values once, then read the Gold runs behind the analytics views.

**Why:** the rule version moves to `analytics-validation-v4` because views 1 and 2 report the D-57 province reporting area. The run-ID recipe of D-52 gains the two province Gold runs, so new province data gives a new ID. Evidence of earlier versions stays as it was. The validation run ID comes from the Gold runs and the rule version, so a rerun on the same Gold data updates the same evidence instead of adding copies.

**Expected result:** one row, with `project_snapshot_count` 1 and `project_snapshot_null_rows` 0.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE validation_rule_version STRING DEFAULT 'analytics-validation-v4';
-- the rule version of 09a_validation_gold_province_reporting that the province gate reads
DECLARE OR REPLACE VARIABLE gold_province_rule_version STRING DEFAULT 'gold-province-reporting-validation-v2';
-- the six views of d-56, in insight order
DECLARE OR REPLACE VARIABLE expected_views ARRAY<STRING> DEFAULT ARRAY(
    'analytics_investment_distribution', 'analytics_population_equity', 'analytics_flood_alignment',
    'analytics_implementation_monitoring', 'analytics_infrastructure_network', 'analytics_integration_coverage'
);
-- the views of the retired d-50, d-51, d-54, and d-55 designs, and the five reporting-area views first built for d-57.
-- analytics_population_equity kept its name
DECLARE OR REPLACE VARIABLE retired_views ARRAY<STRING> DEFAULT ARRAY(
    'analytics_regional_investment', 'analytics_category_portfolio', 'analytics_project_delivery_status',
    'analytics_flood_control_exposure', 'analytics_region_category', 'analytics_infrastructure_year_trend',
    'analytics_infrastructure_year_sector', 'analytics_delivery_cohort', 'analytics_project_size_concentration',
    'analytics_implementing_office_portfolio', 'analytics_region_project_progress',
    'analytics_reporting_area_investment', 'analytics_reporting_area_category',
    'analytics_reporting_area_delivery_status', 'analytics_reporting_area_population_equity',
    'analytics_reporting_area_project_size_concentration'
);
-- the coverage statuses the views may publish. only reported regions get a rank or a ratio
DECLARE OR REPLACE VARIABLE reported_coverage_status STRING DEFAULT 'REPORTED';
DECLARE OR REPLACE VARIABLE limited_coverage_status STRING DEFAULT 'LIMITED_SOURCE_COVERAGE';
DECLARE OR REPLACE VARIABLE no_project_coverage_status STRING DEFAULT 'NO_MAPPED_PROJECT_DATA';
-- the flood point classes that have no single mgb level
DECLARE OR REPLACE VARIABLE unlevelled_flood_classes ARRAY<STRING> DEFAULT ARRAY(
    'AMBIGUOUS', 'OUTSIDE_MAPPED_ZONES', 'NO_USABLE_POINT', 'INVALID_SOURCE'
);
DECLARE OR REPLACE VARIABLE additive_exposure_status STRING DEFAULT 'ADDITIVE_WITHIN_TOLERANCE';
-- the approved status groups (d-40) and the source status d-46 keeps out of the progress indicators
DECLARE OR REPLACE VARIABLE active_status_group STRING DEFAULT 'ACTIVE';
DECLARE OR REPLACE VARIABLE inactive_status_group STRING DEFAULT 'INACTIVE';
DECLARE OR REPLACE VARIABLE for_procurement_source_status STRING DEFAULT 'For Procurement';
-- the eleven integration links of view 6
DECLARE OR REPLACE VARIABLE expected_links ARRAY<STRING> DEFAULT ARRAY(
    'PSGC_REGION', 'PSGC_REPORTING_AREA', 'REGION_PROVINCE_HIERARCHY', 'COORDINATES',
    'POPULATION_VIA_REGION', 'POPULATION_VIA_REPORTING_AREA', 'FLOOD_SUSCEPTIBILITY',
    'PROJECT_CATEGORY', 'PROJECT_STATUS', 'REPORTED_BUDGET', 'OFFICIAL_FLOOD_LIST'
);
-- rounding slack for a share total. a few hundred shares rounded to 4 decimals stay well inside it
DECLARE OR REPLACE VARIABLE share_total_tolerance_pct DOUBLE DEFAULT 0.01;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_context AS
WITH project_fact AS (
    SELECT
        MAX(run_id) AS project_fact_run_id,
        MIN(source_snapshot_id) AS project_snapshot_id,
        MAX(snapshot_date_key) AS snapshot_date_key,
        -- the gate below needs exactly one snapshot and no empty id
        COUNT(DISTINCT source_snapshot_id) AS project_snapshot_count,
        COUNT_IF(source_snapshot_id IS NULL) AS project_snapshot_null_rows
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot
),

region_dimension AS (
    SELECT MAX(run_id) AS dim_region_run_id
    FROM `03-gold`.dim_region
),

population_fact AS (
    SELECT MAX(run_id) AS population_fact_run_id
    FROM `03-gold`.fact_region_population
),

province_area AS (
    -- d-57: the province reporting-area dimension and population behind views 1, 2, 4, 5, and 6
    SELECT MAX(run_id) AS province_area_run_id
    FROM `03-gold`.dim_province_reporting_area
),

province_population AS (
    SELECT MAX(run_id) AS province_population_run_id
    FROM `03-gold`.fact_province_reporting_area_population
),

exposure_fact AS (
    -- the newest exposure run, the same rule view 3 uses
    SELECT
        run_id AS exposure_fact_run_id,
        mgb_source_snapshot_id
    FROM `03-gold`.fact_region_flood_exposure
    QUALIFY ROW_NUMBER() OVER (ORDER BY source_load_ts DESC, run_id) = 1
)

SELECT
    project_fact.*,
    snapshot_day.calendar_date AS snapshot_date,
    region_dimension.dim_region_run_id,
    population_fact.population_fact_run_id,
    exposure_fact.exposure_fact_run_id,
    exposure_fact.mgb_source_snapshot_id,
    province_area.province_area_run_id,
    province_population.province_population_run_id,
    -- the same gold runs always give the same id, so a rerun updates its evidence (d-52).
    -- d-57 adds the two province runs, so new province data gives a new id
    SHA2(CONCAT_WS(
        '|',
        project_fact.project_fact_run_id,
        region_dimension.dim_region_run_id,
        population_fact.population_fact_run_id,
        exposure_fact.exposure_fact_run_id,
        province_area.province_area_run_id,
        province_population.province_population_run_id,
        validation_rule_version
    ), 256) AS validation_run_id
FROM project_fact
CROSS JOIN region_dimension
CROSS JOIN population_fact
CROSS JOIN province_area
CROSS JOIN province_population
LEFT JOIN exposure_fact
    ON TRUE
LEFT JOIN `03-gold`.dim_date AS snapshot_day
    ON project_fact.snapshot_date_key = snapshot_day.date_key;

SELECT * FROM analytics_validation_context;

### One current DPWH snapshot

**What:** stop unless current Gold holds exactly one DPWH snapshot, with no empty snapshot ID.

**Why:** the context reads one snapshot ID, and every check compares the views with it. If current Gold mixed two snapshots, both the views and this notebook would add them together and the checks could still pass. This gate runs before any check uses the context, so no evidence is saved for a mixed snapshot. The analytics notebook runs the same gate before it builds the views.

**Expected result:** the `ASSERT_TRUE` returns `NULL`, and no error is raised.

In [0]:
%sql
SELECT ASSERT_TRUE(
    project_snapshot_count = 1 AND project_snapshot_null_rows = 0,
    'STOP: current Gold project rows span multiple DPWH snapshots. Rebuild Gold before analytics validation.'
)
FROM analytics_validation_context;

## 2. Check that Gold passed its validation

**What:** find the latest Gold validation run that checked the current project, population, and exposure facts together. Stop if any of its STOP checks failed. Then find the latest province Gold validation (`09a_validation_gold_province_reporting`, D-57) of the current project fact, and stop if it is missing or any of its STOP checks failed.

**Why:** analytics must never publish numbers from Gold that failed its own checks. Views 2, 5, and 6 read the population fact, and view 3 reads the exposure fact, so the same Gold validation must cover the population and exposure runs the views read now. The run IDs follow `09_validation_gold`, which records the newest `run_id` of each fact as `pipeline_run_id`.

**Expected result:** `gold_stop_failures` and `province_stop_failures` are 0, the coverage counts are above 0, both `ASSERT_TRUE` calls return `NULL`, and no error is raised.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_gold_gate AS
WITH validated_runs AS (
    -- every validation run scored against the run ids of the facts the views read now,
    -- so a later partial validation run, such as the province one that also checks the
    -- project fact, cannot hide the full gold validation of the same data
    SELECT
        dq.validation_run_id,
        MAX(dq.run_ts) AS latest_run_ts,
        COUNT(DISTINCT dq.table_name) AS covered_current_facts
    FROM `04-validation`.gold_dq_results AS dq
    CROSS JOIN analytics_validation_context AS context
    WHERE (dq.table_name = 'fact_project_snapshot' AND dq.pipeline_run_id = context.project_fact_run_id)
        OR (dq.table_name = 'fact_region_population' AND dq.pipeline_run_id = context.population_fact_run_id)
        OR (dq.table_name = 'fact_region_flood_exposure' AND dq.pipeline_run_id = context.exposure_fact_run_id)
    GROUP BY dq.validation_run_id
),

latest_gold_validation AS (
    -- the latest run that checked the current project, population and exposure facts together
    SELECT validation_run_id
    FROM validated_runs
    WHERE covered_current_facts = 3
    QUALIFY ROW_NUMBER() OVER (ORDER BY latest_run_ts DESC) = 1
),

current_facts AS (
    -- the same run ids 09_validation_gold records for these facts
    SELECT
        (SELECT MAX(run_id) FROM `03-gold`.fact_region_population) AS population_run_id,
        (SELECT MAX(run_id) FROM `03-gold`.fact_region_flood_exposure) AS exposure_run_id
)

SELECT
    COUNT(*) AS gold_checks,
    COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) AS gold_stop_failures,
    COUNT_IF(dq.table_name = 'fact_region_population' AND dq.pipeline_run_id = facts.population_run_id)
        AS current_population_checks,
    COUNT_IF(dq.table_name = 'fact_region_flood_exposure' AND dq.pipeline_run_id = facts.exposure_run_id)
        AS current_exposure_checks,
    MAX(dq.validation_run_id) AS gold_validation_run_id,
    MAX(dq.rule_version) AS gold_rule_version
FROM `04-validation`.gold_dq_results AS dq
INNER JOIN latest_gold_validation AS latest
    ON dq.validation_run_id = latest.validation_run_id
CROSS JOIN current_facts AS facts;

SELECT * FROM analytics_validation_gold_gate;

SELECT ASSERT_TRUE(
    gold_checks > 0
    AND gold_stop_failures = 0
    AND current_population_checks > 0
    AND current_exposure_checks > 0,
    'STOP: Gold validation is missing, failed, or does not cover the current project, population and exposure facts. Run 04_validation/09_validation_gold first.'
)
FROM analytics_validation_gold_gate;

-- d-57: the latest province gold validation of the current project fact must also have passed
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_province_gate AS
WITH latest_province_validation AS (
    SELECT dq.validation_run_id
    FROM `04-validation`.gold_dq_results AS dq
    INNER JOIN analytics_validation_context AS context
        ON dq.pipeline_run_id = context.project_fact_run_id
    WHERE dq.rule_version = gold_province_rule_version
    QUALIFY ROW_NUMBER() OVER (ORDER BY dq.run_ts DESC) = 1
)

SELECT
    COUNT(*) AS province_checks,
    COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) AS province_stop_failures,
    COUNT_IF(dq.table_name = 'fact_province_reporting_area_population') AS province_population_checks,
    COUNT_IF(dq.table_name = 'dim_province_reporting_area') AS province_dimension_checks
FROM `04-validation`.gold_dq_results AS dq
INNER JOIN latest_province_validation AS latest
    ON dq.validation_run_id = latest.validation_run_id;

SELECT * FROM analytics_validation_province_gate;

SELECT ASSERT_TRUE(
    province_checks > 0
    AND province_stop_failures = 0
    AND province_population_checks > 0
    AND province_dimension_checks > 0,
    'STOP: province Gold validation is missing or failed for the current project fact. Run 04_validation/09a_validation_gold_province_reporting first.'
)
FROM analytics_validation_province_gate;

## 3. Read current Gold once, and list the views in `05-analytics`

**What:** build one current-project view from Gold with every field the rebuilds need, and list every view in the `05-analytics` schema.

**Why:** every rebuild below reads the same current rows, so Gold is read once. The flood point class and the status reason are written here in a different form from the views, on purpose. If a view drifts from the documented rule, the comparison breaks. The view list proves that the layer holds exactly the six D-56 views. `CREATE OR REPLACE VIEW` never removes a retired view, so this check catches one that was left behind.

**Expected result:** one row per current project. The view list shows the six expected views and nothing else.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_current AS
SELECT
    fact.project_key,
    fact.region_key,
    fact.status_key,
    fact.reported_budget_pesos,
    fact.physical_progress_pct,
    fact.zero_progress_flag,
    fact.long_running_flag,
    fact.region_match_status,
    fact.coordinate_status,
    fact.flood_match_status,
    fact.latitude,
    fact.longitude,
    fact.start_date_key,
    fact.source_completion_date_key,
    fact.reporting_area_key,
    fact.reporting_area_match_status,
    area.reporting_area_key IS NOT NULL AS area_found,
    area.psgc_version AS area_psgc_version,
    area.region_key AS area_region_key,
    area.reporting_area_name,
    area.reporting_area_type,
    -- written apart from view 5 on purpose
    IF(fact.region_key = 0 OR fact.reporting_area_key = 0, 'NOT_EVALUATED',
        IF(area.region_key <=> fact.region_key, 'CONSISTENT', 'CONFLICT')) AS expected_hierarchy_status,
    region.region_key IS NOT NULL AS region_found,
    region.psgc_version,
    region.is_barmm,
    region.region_name,
    project.project_key IS NOT NULL AS project_found,
    project.contract_id,
    project.infra_year,
    project.standardized_sector,
    project.infra_type_mapping_status,
    project.implementing_office,
    project.is_dpwh_flood_related,
    project.is_in_official_flood_list,
    project.flood_list_match_status,
    status.status_key IS NOT NULL AS status_found,
    status.source_status,
    status.standardized_status,
    status.status_group,
    level.flood_susceptibility_level,
    level.severity_rank,
    -- written apart from the views on purpose, so a view never checks itself
    CASE fact.flood_match_status
        WHEN 'MATCHED' THEN level.flood_susceptibility_level
        WHEN 'UNMATCHED' THEN IF(fact.coordinate_status <=> 'VALID_PAIR', 'OUTSIDE_MAPPED_ZONES', 'NO_USABLE_POINT')
        ELSE fact.flood_match_status
    END AS expected_flood_point_class,
    CASE
        WHEN status.status_group IS NOT NULL THEN 'MAPPED'
        WHEN fact.status_key = 0 THEN 'NO_SOURCE_STATUS'
        ELSE 'NO_APPROVED_STATUS_RULE'
    END AS expected_status_mapping_status
FROM `03-gold`.fact_project_snapshot AS fact
LEFT JOIN `03-gold`.dim_region AS region
    ON fact.region_key = region.region_key
LEFT JOIN `03-gold`.dim_project AS project
    ON fact.project_key = project.project_key
LEFT JOIN `03-gold`.dim_project_status AS status
    ON fact.status_key = status.status_key
LEFT JOIN `03-gold`.dim_flood_susceptibility AS level
    ON fact.flood_susceptibility_key = level.flood_susceptibility_key
LEFT JOIN `03-gold`.dim_province_reporting_area AS area
    ON fact.reporting_area_key = area.reporting_area_key
WHERE fact.is_current_snapshot;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_population AS
WITH newest_year AS (
    SELECT MAX(reference_year) AS reference_year
    FROM `03-gold`.fact_region_population
    WHERE is_primary_source AND region_key <> 0
)

SELECT
    population.region_key,
    population.reference_year,
    population.population_count,
    population.source_name
FROM `03-gold`.fact_region_population AS population
INNER JOIN newest_year
    ON population.reference_year = newest_year.reference_year
WHERE population.is_primary_source
    AND population.region_key <> 0;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_area_population AS
WITH newest_year AS (
    SELECT MAX(reference_year) AS reference_year
    FROM `03-gold`.fact_province_reporting_area_population
    WHERE is_primary_source AND reporting_area_key <> 0
)

SELECT
    population.reporting_area_key,
    population.reference_year,
    population.population_count,
    population.source_name
FROM `03-gold`.fact_province_reporting_area_population AS population
INNER JOIN newest_year
    ON population.reference_year = newest_year.reference_year
WHERE population.is_primary_source
    AND population.reporting_area_key <> 0;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_layer AS
SELECT
    views.table_name,
    ARRAY_CONTAINS(expected_views, views.table_name) AS is_expected,
    ARRAY_CONTAINS(retired_views, views.table_name) AS is_retired
FROM information_schema.views AS views
WHERE views.table_schema = '05-analytics';

SELECT * FROM analytics_validation_layer ORDER BY is_expected DESC, table_name;

## 4. Insight 1: rebuild `analytics_investment_distribution` from Gold

**What:** rebuild each level, member, and cohort row, the member totals, shares, ranks, and running shares, then compare them with the view.

**Why:** the rebuild counts rows with `COUNT(*)`, while the view counts distinct projects. A separate check stops the run if a project has more than one current row. Budgets compare with `<=>`, so a `NULL` budget that turned into `0` is a break. The region level is rebuilt from `region_key` and the reporting-area level from `reporting_area_key`, each on its own, so a region total can never be rebuilt from provinces. Views 2 and 3 take their members from view 1, so view 1 is checked against Gold first.

**Expected result:** one compared row per level, member, and cohort, and no break columns set to `true`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_inv_output AS
SELECT * FROM `05-analytics`.analytics_investment_distribution;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_inv_members AS
WITH region_versions AS (
    SELECT DISTINCT psgc_version
    FROM analytics_validation_current
    WHERE region_key <> 0 AND region_found
),

area_versions AS (
    SELECT DISTINCT area_psgc_version AS psgc_version
    FROM analytics_validation_current
    WHERE reporting_area_key <> 0 AND area_found
),

boundaries AS (
    SELECT
        reporting_area_key,
        MAX_BY(boundary_status, NAMED_STRUCT('loaded', source_load_ts, 'version', boundary_version)) AS boundary_status
    FROM `03-gold`.dim_province_reporting_area_boundary
    GROUP BY reporting_area_key
)

SELECT
    'REGION' AS geography_level,
    region.region_key AS geography_key,
    region.region_name AS geography_name,
    'REGION' AS geography_type,
    region.region_key AS parent_region_key,
    CAST(NULL AS STRING) AS boundary_status,
    region.is_barmm
FROM `03-gold`.dim_region AS region
INNER JOIN region_versions AS version
    ON region.psgc_version <=> version.psgc_version
WHERE region.region_key <> 0
UNION ALL
SELECT
    'REPORTING_AREA' AS geography_level,
    area.reporting_area_key AS geography_key,
    area.reporting_area_name AS geography_name,
    area.reporting_area_type AS geography_type,
    area.region_key AS parent_region_key,
    boundaries.boundary_status,
    COALESCE(parent.is_barmm, FALSE) AS is_barmm
FROM `03-gold`.dim_province_reporting_area AS area
INNER JOIN area_versions AS version
    ON area.psgc_version <=> version.psgc_version
LEFT JOIN `03-gold`.dim_region AS parent
    ON area.region_key = parent.region_key
LEFT JOIN boundaries
    ON area.reporting_area_key = boundaries.reporting_area_key
WHERE area.reporting_area_key <> 0;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_inv_expected AS
WITH assignments AS (
    -- each level is rebuilt from its own gold key
    SELECT
        'REGION' AS geography_level,
        region_key AS geography_key,
        infra_year,
        reported_budget_pesos
    FROM analytics_validation_current
    WHERE region_key <> 0 AND region_found
    UNION ALL
    SELECT
        'REPORTING_AREA' AS geography_level,
        reporting_area_key AS geography_key,
        infra_year,
        reported_budget_pesos
    FROM analytics_validation_current
    WHERE reporting_area_key <> 0 AND area_found
),

cohorts AS (
    SELECT
        geography_level,
        geography_key,
        infra_year,
        COUNT(*) AS project_count,
        COUNT_IF(reported_budget_pesos IS NOT NULL) AS projects_with_budget,
        SUM(reported_budget_pesos) AS budget
    FROM assignments
    GROUP BY ALL
),

member_totals AS (
    SELECT
        geography_level,
        geography_key,
        SUM(project_count) AS member_projects,
        SUM(budget) AS member_budget
    FROM cohorts
    GROUP BY ALL
),

member_status AS (
    SELECT
        members.*,
        COALESCE(member_totals.member_projects, 0) AS member_projects,
        member_totals.member_budget,
        CASE
            WHEN COALESCE(member_totals.member_projects, 0) = 0 THEN no_project_coverage_status
            WHEN members.is_barmm THEN limited_coverage_status
            ELSE reported_coverage_status
        END AS coverage_status
    FROM analytics_validation_inv_members AS members
    LEFT JOIN member_totals
        ON members.geography_level = member_totals.geography_level
        AND members.geography_key = member_totals.geography_key
),

member_ranked AS (
    SELECT
        member_status.*,
        ROUND(
            100.0 * member_budget / NULLIF(SUM(member_budget) OVER (PARTITION BY geography_level), 0), 4
        ) AS member_share,
        IF(coverage_status = reported_coverage_status, RANK() OVER (
            PARTITION BY geography_level, coverage_status = reported_coverage_status
            ORDER BY member_budget DESC NULLS LAST
        ), NULL) AS member_budget_rank,
        IF(coverage_status = reported_coverage_status, RANK() OVER (
            PARTITION BY geography_level, coverage_status = reported_coverage_status
            ORDER BY member_projects DESC
        ), NULL) AS member_project_rank,
        IF(coverage_status = reported_coverage_status AND member_budget IS NOT NULL, ROUND(
            100.0 * SUM(member_budget) OVER (
                PARTITION BY geography_level, coverage_status = reported_coverage_status
                ORDER BY member_budget DESC NULLS LAST, geography_key
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) / NULLIF(SUM(member_budget) OVER (PARTITION BY geography_level), 0),
            4
        ), NULL) AS member_cumulative
    FROM member_status
),

cohort_rows AS (
    SELECT
        member_ranked.*,
        cohorts.infra_year,
        CASE
            WHEN cohorts.geography_key IS NULL THEN 'NO_MAPPED_PROJECT'
            WHEN cohorts.infra_year IS NULL THEN 'NO_INFRA_YEAR'
            ELSE 'INFRA_YEAR'
        END AS row_type,
        COALESCE(cohorts.project_count, 0) AS project_count,
        COALESCE(cohorts.projects_with_budget, 0) AS projects_with_budget,
        cohorts.budget
    FROM member_ranked
    LEFT JOIN cohorts
        ON member_ranked.geography_level = cohorts.geography_level
        AND member_ranked.geography_key = cohorts.geography_key
)

SELECT
    cohort_rows.*,
    ROUND(budget / NULLIF(projects_with_budget, 0), 2) AS avg_budget,
    ROUND(
        100.0 * budget / NULLIF(SUM(budget) OVER (PARTITION BY geography_level, infra_year), 0), 4
    ) AS cohort_share,
    IF(coverage_status = reported_coverage_status, RANK() OVER (
        PARTITION BY geography_level, infra_year, coverage_status = reported_coverage_status
        ORDER BY budget DESC NULLS LAST
    ), NULL) AS cohort_budget_rank,
    IF(coverage_status = reported_coverage_status, RANK() OVER (
        PARTITION BY geography_level, infra_year, coverage_status = reported_coverage_status
        ORDER BY project_count DESC
    ), NULL) AS cohort_project_rank,
    IF(coverage_status = reported_coverage_status AND budget IS NOT NULL, ROUND(
        100.0 * SUM(budget) OVER (
            PARTITION BY geography_level, infra_year, coverage_status = reported_coverage_status
            ORDER BY budget DESC NULLS LAST, geography_key
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) / NULLIF(SUM(budget) OVER (PARTITION BY geography_level, infra_year), 0),
        4
    ), NULL) AS cohort_cumulative
FROM cohort_rows;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_inv_compare AS
SELECT
    COALESCE(expected.geography_level, output.geography_level) AS geography_level,
    COALESCE(expected.geography_key, output.geography_key) AS geography_key,
    COALESCE(expected.geography_name, output.geography_name) AS geography_name,
    COALESCE(expected.infra_year, output.infra_year) AS infra_year,
    expected.is_barmm,
    output.source_coverage_status,
    expected.geography_key IS NULL AS extra_in_output,
    output.geography_key IS NULL AS missing_from_output,
    NOT (output.cohort_row_type <=> expected.row_type) AS row_type_break,
    NOT (output.geography_type <=> expected.geography_type)
    OR NOT (output.parent_region_key <=> expected.parent_region_key)
    OR NOT (output.boundary_status <=> expected.boundary_status) AS member_break,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.projects_with_budget <=> expected.projects_with_budget)
    OR NOT (output.total_reported_budget_pesos <=> expected.budget) AS count_budget_break,
    NOT (output.avg_budget_per_project_pesos <=> expected.avg_budget) AS average_break,
    NOT (output.source_coverage_status <=> expected.coverage_status) AS status_break,
    NOT (output.cohort_budget_share_pct <=> expected.cohort_share) AS cohort_share_break,
    NOT (output.cohort_budget_rank <=> expected.cohort_budget_rank)
    OR NOT (output.cohort_project_count_rank <=> expected.cohort_project_rank) AS cohort_rank_break,
    NOT (output.cohort_cumulative_budget_share_pct <=> expected.cohort_cumulative) AS cohort_cumulative_break,
    NOT (output.geography_project_count <=> expected.member_projects)
    OR NOT (output.geography_budget_pesos <=> expected.member_budget) AS member_total_break,
    NOT (output.geography_budget_share_pct <=> expected.member_share)
    OR NOT (output.geography_budget_rank <=> expected.member_budget_rank)
    OR NOT (output.geography_project_count_rank <=> expected.member_project_rank)
    OR NOT (output.geography_cumulative_budget_share_pct <=> expected.member_cumulative) AS member_rank_break,
    output.project_count = 0 AND output.total_reported_budget_pesos IS NOT NULL AS budget_without_projects
FROM analytics_validation_inv_expected AS expected
FULL OUTER JOIN analytics_validation_inv_output AS output
    ON expected.geography_level = output.geography_level
    AND expected.geography_key = output.geography_key
    AND expected.infra_year <=> output.infra_year;

SELECT * FROM analytics_validation_inv_compare ORDER BY geography_level, geography_name, infra_year;

## 5. Insight 2: rebuild `analytics_population_equity` from Gold

**What:** rebuild each province reporting area from the Gold member totals of step 4 and the newest primary reporting-area population, then compare it with the view.

**Why:** both shares in the gap must cover the same comparable areas, and an area outside that set must show no share, gap, ratio, or per-capita value. Per-person ratios must leave key `0` out (#61). The view's population must add up to the Gold primary population of every geographic reporting area, so no area's population is lost or counted twice.

**Expected result:** no break columns set to `true`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_eq_output AS
SELECT * FROM `05-analytics`.analytics_population_equity;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_eq_expected AS
WITH areas AS (
    SELECT DISTINCT
        geography_key AS reporting_area_key,
        parent_region_key,
        coverage_status,
        member_projects,
        member_budget
    FROM analytics_validation_inv_expected
    WHERE geography_level = 'REPORTING_AREA'
),

combined AS (
    SELECT
        areas.*,
        population.reference_year,
        population.population_count,
        population.source_name,
        CASE
            WHEN areas.coverage_status <> reported_coverage_status THEN 'EXCLUDED_SOURCE_COVERAGE'
            WHEN areas.member_budget IS NULL THEN 'EXCLUDED_NO_REPORTED_BUDGET'
            WHEN population.population_count IS NULL THEN 'EXCLUDED_NO_POPULATION'
            ELSE 'COMPARABLE'
        END AS comparison_status
    FROM areas
    LEFT JOIN analytics_validation_area_population AS population
        ON areas.reporting_area_key = population.reporting_area_key
),

shares AS (
    SELECT
        combined.*,
        IF(comparison_status = 'COMPARABLE', member_budget, NULL) AS comparable_budget,
        IF(comparison_status = 'COMPARABLE', population_count, NULL) AS comparable_population
    FROM combined
)

SELECT
    shares.*,
    ROUND(100.0 * population_count / NULLIF(SUM(population_count) OVER (), 0), 4) AS population_share,
    ROUND(100.0 * comparable_population / NULLIF(SUM(comparable_population) OVER (), 0), 4)
        AS comparable_population_share,
    ROUND(100.0 * comparable_budget / NULLIF(SUM(comparable_budget) OVER (), 0), 4) AS investment_share,
    ROUND(comparable_budget / NULLIF(comparable_population, 0), 2) AS per_capita,
    IF(comparison_status = 'COMPARABLE', ROUND(
        SUM(comparable_budget) OVER () / NULLIF(SUM(comparable_population) OVER (), 0), 2
    ), NULL) AS set_per_capita
FROM shares;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_eq_compare AS
WITH ranked AS (
    SELECT
        analytics_validation_eq_expected.*,
        IF(comparison_status = 'COMPARABLE', RANK() OVER (
            PARTITION BY comparison_status = 'COMPARABLE'
            ORDER BY per_capita DESC
        ), NULL) AS per_capita_rank
    FROM analytics_validation_eq_expected
)

SELECT
    COALESCE(expected.reporting_area_key, output.reporting_area_key) AS reporting_area_key,
    output.reporting_area_name,
    expected.comparison_status,
    expected.reporting_area_key IS NULL OR output.reporting_area_key IS NULL AS area_set_break,
    NOT (output.source_coverage_status <=> expected.coverage_status)
    OR NOT (output.region_key <=> expected.parent_region_key)
    OR NOT (output.project_count <=> expected.member_projects)
    OR NOT (output.total_reported_budget_pesos <=> expected.member_budget) AS area_total_break,
    NOT (output.population_reference_year <=> expected.reference_year)
    OR NOT (output.population_count <=> expected.population_count)
    OR NOT (output.population_source_name <=> expected.source_name) AS population_break,
    NOT (output.equity_comparison_status <=> expected.comparison_status) AS comparison_status_break,
    NOT (output.population_share_pct <=> expected.population_share)
    OR NOT (output.comparable_population_share_pct <=> expected.comparable_population_share)
    OR NOT (output.investment_share_pct <=> expected.investment_share) AS share_break,
    NOT (output.share_gap_pp <=> expected.investment_share - expected.comparable_population_share)
    OR NOT (output.investment_to_population_share_ratio <=> ROUND(
        expected.investment_share / NULLIF(expected.comparable_population_share, 0), 4
    )) AS gap_break,
    NOT (output.investment_per_capita_pesos <=> expected.per_capita)
    OR NOT (output.investment_per_capita_rank <=> expected.per_capita_rank)
    OR NOT (output.comparable_set_per_capita_pesos <=> expected.set_per_capita) AS per_capita_break,
    expected.comparison_status <> 'COMPARABLE'
    AND (
        output.investment_share_pct IS NOT NULL
        OR output.comparable_population_share_pct IS NOT NULL
        OR output.share_gap_pp IS NOT NULL
        OR output.investment_to_population_share_ratio IS NOT NULL
        OR output.investment_per_capita_pesos IS NOT NULL
        OR output.investment_per_capita_rank IS NOT NULL
    ) AS not_comparable_ratio_break,
    output.population_count IS NULL AS missing_population,
    output.investment_share_pct,
    output.population_share_pct,
    output.comparable_population_share_pct
FROM ranked AS expected
FULL OUTER JOIN analytics_validation_eq_output AS output
    ON expected.reporting_area_key = output.reporting_area_key;

SELECT * FROM analytics_validation_eq_compare ORDER BY reporting_area_key;

## 6. Insight 3: rebuild `analytics_flood_alignment` from Gold

**What:** rebuild the region and flood point class grid, every project count and budget, and the area of each MGB level from the newest Gold exposure run, then compare them with the view.

**Why:** the view must reuse the Gold flood result, never repeat spatial work, and never hide a project. So every current project outside key `0` must land in exactly one published class. Flood-control, other, and uncategorized must add up to the row's project count. Each region's classes must add up to its view 1 project count. Area may appear only on a level row. A region with no mapped DPWH project must keep `NULL` project figures.

**Expected result:** no break columns set to `true`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_fa_output AS
SELECT * FROM `05-analytics`.analytics_flood_alignment;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_fa_classes AS
SELECT
    flood_susceptibility_level AS flood_point_class,
    TRUE AS is_mgb_level,
    MAX(severity_rank) AS severity_rank
FROM `03-gold`.dim_flood_susceptibility
WHERE flood_susceptibility_key <> 0
GROUP BY ALL
UNION ALL
SELECT
    EXPLODE(unlevelled_flood_classes) AS flood_point_class,
    FALSE AS is_mgb_level,
    CAST(NULL AS INT) AS severity_rank;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_fa_expected AS
WITH regions AS (
    SELECT DISTINCT
        geography_key AS region_key,
        coverage_status
    FROM analytics_validation_inv_expected
    WHERE geography_level = 'REGION'
),

projects AS (
    SELECT
        region_key,
        expected_flood_point_class AS flood_point_class,
        COUNT(*) AS project_count,
        SUM(reported_budget_pesos) AS budget,
        COUNT_IF(is_dpwh_flood_related) AS flood_control_projects,
        SUM(IF(is_dpwh_flood_related, reported_budget_pesos, NULL)) AS flood_control_budget,
        COUNT_IF(standardized_sector = 'Flood Control and Drainage') AS strict_projects,
        SUM(IF(standardized_sector = 'Flood Control and Drainage', reported_budget_pesos, NULL)) AS strict_budget,
        COUNT_IF(is_in_official_flood_list) AS list_projects,
        SUM(IF(is_in_official_flood_list, reported_budget_pesos, NULL)) AS list_budget,
        COUNT_IF(NOT is_dpwh_flood_related) AS other_projects,
        COUNT_IF(is_dpwh_flood_related IS NULL) AS uncategorized_projects
    FROM analytics_validation_current
    WHERE region_key <> 0 AND region_found
    GROUP BY ALL
),

areas AS (
    SELECT
        exposure.region_key,
        level.flood_susceptibility_level AS flood_point_class,
        exposure.susceptible_area_sqkm,
        exposure.share_of_region_area_pct,
        exposure.level_area_additivity_status
    FROM `03-gold`.fact_region_flood_exposure AS exposure
    INNER JOIN analytics_validation_context AS context
        ON exposure.run_id = context.exposure_fact_run_id
    INNER JOIN `03-gold`.dim_flood_susceptibility AS level
        ON exposure.flood_susceptibility_key = level.flood_susceptibility_key
    WHERE exposure.flood_susceptibility_key <> 0
),

area_status AS (
    SELECT
        region_key,
        CASE
            WHEN MAX(level_area_additivity_status = 'NOT_EVALUATED') THEN 'NOT_EVALUATED'
            WHEN MIN(level_area_additivity_status <=> additive_exposure_status) THEN additive_exposure_status
            ELSE 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP'
        END AS exposure_area_status
    FROM areas
    GROUP BY ALL
)

SELECT
    regions.region_key,
    regions.coverage_status,
    classes.flood_point_class,
    classes.is_mgb_level,
    classes.severity_rank,
    regions.coverage_status = no_project_coverage_status AS is_unknown_region,
    IF(regions.coverage_status = no_project_coverage_status, NULL, COALESCE(projects.project_count, 0)) AS project_count,
    projects.budget,
    IF(regions.coverage_status = no_project_coverage_status, NULL, COALESCE(projects.flood_control_projects, 0))
        AS flood_control_projects,
    projects.flood_control_budget,
    IF(regions.coverage_status = no_project_coverage_status, NULL, COALESCE(projects.strict_projects, 0))
        AS strict_projects,
    projects.strict_budget,
    IF(regions.coverage_status = no_project_coverage_status, NULL, COALESCE(projects.list_projects, 0))
        AS list_projects,
    projects.list_budget,
    IF(regions.coverage_status = no_project_coverage_status, NULL, COALESCE(projects.other_projects, 0))
        AS other_projects,
    IF(regions.coverage_status = no_project_coverage_status, NULL, COALESCE(projects.uncategorized_projects, 0))
        AS uncategorized_projects,
    areas.susceptible_area_sqkm,
    areas.share_of_region_area_pct,
    COALESCE(area_status.exposure_area_status, 'NO_EXPOSURE_DATA') AS exposure_area_status
FROM regions
CROSS JOIN analytics_validation_fa_classes AS classes
LEFT JOIN projects
    ON regions.region_key = projects.region_key
    AND classes.flood_point_class = projects.flood_point_class
LEFT JOIN areas
    ON regions.region_key = areas.region_key
    AND classes.flood_point_class = areas.flood_point_class
LEFT JOIN area_status
    ON regions.region_key = area_status.region_key;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_fa_compare AS
SELECT
    COALESCE(expected.region_key, output.region_key) AS region_key,
    COALESCE(expected.flood_point_class, output.flood_point_class) AS flood_point_class,
    expected.is_mgb_level,
    expected.is_unknown_region,
    expected.region_key IS NULL AS extra_in_output,
    output.region_key IS NULL AS missing_from_output,
    NOT (output.is_mgb_level <=> expected.is_mgb_level)
    OR NOT (output.severity_rank <=> expected.severity_rank)
    OR NOT (output.source_coverage_status <=> expected.coverage_status) AS class_break,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.total_reported_budget_pesos <=> expected.budget) AS count_budget_break,
    NOT (output.flood_control_project_count <=> expected.flood_control_projects)
    OR NOT (output.flood_control_budget_pesos <=> expected.flood_control_budget)
    OR NOT (output.flood_control_single_sector_project_count <=> expected.strict_projects)
    OR NOT (output.flood_control_single_sector_budget_pesos <=> expected.strict_budget)
    OR NOT (output.official_flood_list_project_count <=> expected.list_projects)
    OR NOT (output.official_flood_list_budget_pesos <=> expected.list_budget) AS flood_measure_break,
    NOT (output.other_project_count <=> expected.other_projects)
    OR NOT (output.uncategorized_project_count <=> expected.uncategorized_projects) AS other_measure_break,
    NOT (output.project_count <=> output.flood_control_project_count + output.other_project_count
        + output.uncategorized_project_count) AS partition_break,
    NOT (output.susceptible_area_sqkm <=> expected.susceptible_area_sqkm)
    OR NOT (output.share_of_region_area_pct <=> expected.share_of_region_area_pct)
    OR NOT (output.exposure_area_status <=> expected.exposure_area_status) AS area_break,
    NOT expected.is_mgb_level AND (output.susceptible_area_sqkm IS NOT NULL
        OR output.share_of_region_area_pct IS NOT NULL) AS area_on_class_break,
    expected.is_unknown_region AND (output.project_count IS NOT NULL
        OR output.flood_control_project_count IS NOT NULL
        OR output.total_reported_budget_pesos IS NOT NULL) AS unknown_region_break,
    output.flood_control_project_count,
    output.flood_control_budget_pesos,
    output.share_of_region_flood_control_projects_pct,
    output.share_of_region_flood_control_budget_pct,
    output.exposure_area_status
FROM analytics_validation_fa_expected AS expected
FULL OUTER JOIN analytics_validation_fa_output AS output
    ON expected.region_key = output.region_key
    AND expected.flood_point_class <=> output.flood_point_class;

SELECT * FROM analytics_validation_fa_compare ORDER BY region_key, flood_point_class;

## 7. Insight 4: rebuild `analytics_implementation_monitoring` from Gold

**What:** rebuild every project row, its status reason, progress reporting status, dates, and the five review indicators, then compare them with the view by `project_key`.

**Why:** each review indicator is rebuilt with `IF` and `COALESCE` instead of the view's expressions, so a drift in either breaks the comparison. Zero reported progress, progress that is not published, `For Procurement`, inactive, and unresolved statuses must stay separate. A `For Procurement` project must never raise a progress indicator (D-46). The view must hold every current Gold project once, with its Gold budget.

**Expected result:** no break columns set to `true`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_im_output AS
SELECT * FROM `05-analytics`.analytics_implementation_monitoring;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_im_expected AS
WITH projects AS (
    SELECT
        gold_project.*,
        COALESCE(gold_project.source_status = for_procurement_source_status, FALSE) AS for_procurement,
        IF(gold_project.physical_progress_pct IS NULL, 'NOT_PUBLISHED',
            IF(gold_project.physical_progress_pct > 0, 'ABOVE_ZERO', 'ZERO')) AS progress_status,
        start_day.calendar_date AS start_date,
        completion_day.calendar_date AS completion_date
    FROM analytics_validation_current AS gold_project
    LEFT JOIN `03-gold`.dim_date AS start_day
        ON gold_project.start_date_key = start_day.date_key
    LEFT JOIN `03-gold`.dim_date AS completion_day
        ON gold_project.source_completion_date_key = completion_day.date_key
),

indicators AS (
    SELECT
        projects.*,
        IF(status_group <=> active_status_group AND NOT for_procurement,
            COALESCE(zero_progress_flag, FALSE), FALSE) AS active_zero,
        IF(status_group <=> active_status_group, COALESCE(long_running_flag, FALSE), FALSE) AS active_long,
        status_group <=> active_status_group AND NOT for_procurement
        AND physical_progress_pct IS NULL AS active_not_published,
        status_group <=> inactive_status_group AS inactive,
        status_group IS NULL AS unresolved
    FROM projects
)

SELECT
    indicators.*,
    IF(active_zero, 1, 0) + IF(active_long, 1, 0) + IF(active_not_published, 1, 0)
    + IF(inactive, 1, 0) + IF(unresolved, 1, 0) AS reason_count
FROM indicators;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_im_compare AS
SELECT
    COALESCE(expected.project_key, output.project_key) AS project_key,
    expected.project_key IS NULL AS extra_in_output,
    output.project_key IS NULL AS missing_from_output,
    NOT (output.contract_id <=> expected.contract_id)
    OR NOT (output.region_key <=> expected.region_key)
    OR NOT (output.region_name <=> expected.region_name)
    OR NOT (output.region_match_status <=> expected.region_match_status)
    OR NOT (output.reporting_area_key <=> expected.reporting_area_key)
    OR NOT (output.reporting_area_name <=> expected.reporting_area_name)
    OR NOT (output.reporting_area_type <=> expected.reporting_area_type)
    OR NOT (output.reporting_area_match_status <=> expected.reporting_area_match_status)
    OR NOT (output.standardized_sector <=> expected.standardized_sector)
    OR NOT (output.infra_type_mapping_status <=> expected.infra_type_mapping_status)
    OR NOT (output.implementing_office <=> expected.implementing_office)
    OR NOT (output.infra_year <=> expected.infra_year) AS attribute_break,
    NOT (output.source_status <=> expected.source_status)
    OR NOT (output.standardized_status <=> expected.standardized_status)
    OR NOT (output.status_group <=> expected.status_group)
    OR NOT (output.status_mapping_status <=> expected.expected_status_mapping_status)
    OR NOT (output.is_for_procurement <=> expected.for_procurement) AS status_break,
    NOT (output.start_date <=> expected.start_date)
    OR NOT (output.source_completion_date <=> expected.completion_date) AS date_break,
    NOT (output.physical_progress_pct <=> expected.physical_progress_pct)
    OR NOT (output.progress_reporting_status <=> expected.progress_status)
    OR NOT (output.zero_progress_flag <=> expected.zero_progress_flag)
    OR NOT (output.long_running_flag <=> expected.long_running_flag)
    OR NOT (output.reported_budget_pesos <=> expected.reported_budget_pesos) AS measure_break,
    NOT (output.review_active_zero_progress <=> expected.active_zero)
    OR NOT (output.review_active_long_running <=> expected.active_long)
    OR NOT (output.review_active_progress_not_published <=> expected.active_not_published)
    OR NOT (output.review_inactive_status <=> expected.inactive)
    OR NOT (output.review_unresolved_status <=> expected.unresolved)
    OR NOT (output.review_reason_count <=> expected.reason_count)
    OR NOT (output.needs_review <=> (expected.reason_count > 0)) AS indicator_break,
    output.is_for_procurement
    AND (output.review_active_zero_progress OR output.review_active_progress_not_published) AS procurement_raised,
    (output.progress_reporting_status = 'NOT_PUBLISHED') <> (output.physical_progress_pct IS NULL) AS unpublished_break,
    expected.zero_progress_flag IS NOT NULL
    AND expected.physical_progress_pct IS NOT NULL
    AND expected.zero_progress_flag <> (expected.physical_progress_pct = 0) AS progress_flag_disagrees,
    expected.zero_progress_flag IS NOT NULL
    AND expected.physical_progress_pct IS NULL AS flag_without_published_progress,
    output.needs_review,
    output.review_active_zero_progress,
    output.review_active_long_running,
    output.review_active_progress_not_published,
    output.review_inactive_status,
    output.review_unresolved_status,
    output.reported_budget_pesos AS output_budget
FROM analytics_validation_im_expected AS expected
FULL OUTER JOIN analytics_validation_im_output AS output
    ON expected.project_key = output.project_key;

SELECT
    COUNT(*) AS compared_projects,
    COUNT_IF(extra_in_output OR missing_from_output) AS project_set_breaks,
    COUNT_IF(attribute_break OR status_break OR date_break OR measure_break) AS field_breaks,
    COUNT_IF(indicator_break) AS indicator_breaks,
    COUNT_IF(needs_review) AS projects_needing_review
FROM analytics_validation_im_compare;

## 8. Insight 5: rebuild `analytics_infrastructure_network` from Gold

**What:** rebuild every project row with its point, display cell, flood point class, status reason, and region population, then compare them with the view by `project_key`.

**Why:** the map must show Gold points as published, mark only valid pairs as mappable, and put every mappable point inside its own 0.1-degree cell. The population context must be the newest primary population of the project's region, and key `0` must have none.

**Expected result:** no break columns set to `true`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_net_output AS
SELECT * FROM `05-analytics`.analytics_infrastructure_network;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_net_compare AS
WITH expected AS (
    SELECT
        gold_project.*,
        gold_project.coordinate_status <=> 'VALID_PAIR' AS mappable,
        population.population_count,
        population.reference_year,
        area_population.population_count AS area_population_count
    FROM analytics_validation_current AS gold_project
    LEFT JOIN analytics_validation_population AS population
        ON gold_project.region_key = population.region_key
    LEFT JOIN analytics_validation_area_population AS area_population
        ON gold_project.reporting_area_key = area_population.reporting_area_key
)

SELECT
    COALESCE(expected.project_key, output.project_key) AS project_key,
    expected.project_key IS NULL AS extra_in_output,
    output.project_key IS NULL AS missing_from_output,
    NOT (output.contract_id <=> expected.contract_id)
    OR NOT (output.region_key <=> expected.region_key)
    OR NOT (output.region_name <=> expected.region_name)
    OR NOT (output.region_match_status <=> expected.region_match_status)
    OR NOT (output.reporting_area_key <=> expected.reporting_area_key)
    OR NOT (output.reporting_area_name <=> expected.reporting_area_name)
    OR NOT (output.reporting_area_type <=> expected.reporting_area_type)
    OR NOT (output.reporting_area_match_status <=> expected.reporting_area_match_status)
    OR NOT (output.standardized_sector <=> expected.standardized_sector)
    OR NOT (output.infra_type_mapping_status <=> expected.infra_type_mapping_status)
    OR NOT (output.is_dpwh_flood_related <=> expected.is_dpwh_flood_related)
    OR NOT (output.is_in_official_flood_list <=> expected.is_in_official_flood_list)
    OR NOT (output.infra_year <=> expected.infra_year)
    OR NOT (output.status_group <=> expected.status_group)
    OR NOT (output.status_mapping_status <=> expected.expected_status_mapping_status)
    OR NOT (output.reported_budget_pesos <=> expected.reported_budget_pesos) AS attribute_break,
    NOT (output.latitude <=> expected.latitude)
    OR NOT (output.longitude <=> expected.longitude)
    OR NOT (output.coordinate_status <=> expected.coordinate_status)
    OR NOT (output.is_mappable <=> expected.mappable) AS point_break,
    -- a mappable point sits inside its own cell. any other row has no cell
    NOT IF(
        expected.mappable,
        output.latitude >= output.map_grid_latitude AND output.latitude < output.map_grid_latitude + 0.1
        AND output.longitude >= output.map_grid_longitude AND output.longitude < output.map_grid_longitude + 0.1,
        output.map_grid_latitude IS NULL AND output.map_grid_longitude IS NULL
    ) AS grid_break,
    NOT (output.flood_point_class <=> expected.expected_flood_point_class)
    OR NOT (output.flood_severity_rank <=> IF(expected.flood_match_status = 'MATCHED', expected.severity_rank, NULL))
        AS flood_break,
    NOT (output.region_population_count <=> expected.population_count)
    OR NOT (output.population_reference_year <=> expected.reference_year)
    OR NOT (output.reporting_area_population_count <=> expected.area_population_count) AS population_break,
    NOT (output.region_province_hierarchy_status <=> expected.expected_hierarchy_status) AS hierarchy_break,
    expected.expected_hierarchy_status = 'CONFLICT' AS hierarchy_conflict,
    expected.region_key = 0 AND output.region_population_count IS NOT NULL AS key0_population_break,
    expected.mappable AS is_mappable,
    expected.mappable AND expected.region_key = 0 AS mappable_without_region,
    output.reported_budget_pesos AS output_budget
FROM expected
FULL OUTER JOIN analytics_validation_net_output AS output
    ON expected.project_key = output.project_key;

SELECT
    COUNT(*) AS compared_projects,
    COUNT_IF(extra_in_output OR missing_from_output) AS project_set_breaks,
    COUNT_IF(attribute_break OR point_break OR grid_break OR flood_break OR population_break) AS field_breaks,
    COUNT_IF(NOT is_mappable) AS projects_not_mappable
FROM analytics_validation_net_compare;

## 9. Insight 6: rebuild `analytics_integration_coverage` from Gold

**What:** rebuild every link, outcome, and region row from Gold with one `UNION ALL` branch per link, then compare them with the view.

**Why:** the view builds its links in one pass with `INLINE`. This rebuild uses separate branches, so the two never share code. Each link must hold every current project exactly once, so matched and unmatched projects and budgets add up to Gold. `is_linked` must follow the documented rule of each link.

**Expected result:** no break columns set to `true`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_ic_output AS
SELECT * FROM `05-analytics`.analytics_integration_coverage;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_ic_expected AS
WITH projects AS (
    SELECT
        gold_project.*,
        population.region_key IS NOT NULL AS has_population,
        area_population.population_count IS NOT NULL AS has_area_population
    FROM analytics_validation_current AS gold_project
    LEFT JOIN analytics_validation_population AS population
        ON gold_project.region_key = population.region_key
    LEFT JOIN analytics_validation_area_population AS area_population
        ON gold_project.reporting_area_key = area_population.reporting_area_key
),

links AS (
    SELECT
        'PSGC_REGION' AS integration_link,
        region_match_status AS link_outcome,
        region_key <> 0 AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'PSGC_REPORTING_AREA' AS integration_link,
        reporting_area_match_status AS link_outcome,
        reporting_area_key <> 0 AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'REGION_PROVINCE_HIERARCHY' AS integration_link,
        expected_hierarchy_status AS link_outcome,
        expected_hierarchy_status = 'CONSISTENT' AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'POPULATION_VIA_REPORTING_AREA' AS integration_link,
        IF(reporting_area_key = 0, 'NO_REPORTING_AREA',
            IF(has_area_population, 'LINKED', 'REPORTING_AREA_WITHOUT_POPULATION')) AS link_outcome,
        reporting_area_key <> 0 AND has_area_population AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'COORDINATES' AS integration_link,
        coordinate_status AS link_outcome,
        COALESCE(coordinate_status = 'VALID_PAIR', FALSE) AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'POPULATION_VIA_REGION' AS integration_link,
        IF(region_key = 0, 'NO_OFFICIAL_REGION', IF(has_population, 'LINKED', 'REGION_WITHOUT_POPULATION'))
            AS link_outcome,
        region_key <> 0 AND has_population AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'FLOOD_SUSCEPTIBILITY' AS integration_link,
        IF(flood_match_status = 'MATCHED', 'MATCHED', expected_flood_point_class) AS link_outcome,
        COALESCE(flood_match_status = 'MATCHED', FALSE) AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'PROJECT_CATEGORY' AS integration_link,
        infra_type_mapping_status AS link_outcome,
        standardized_sector IS NOT NULL AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'PROJECT_STATUS' AS integration_link,
        expected_status_mapping_status AS link_outcome,
        status_group IS NOT NULL AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'REPORTED_BUDGET' AS integration_link,
        IF(reported_budget_pesos IS NULL, 'NOT_REPORTED',
            IF(reported_budget_pesos > 0, 'REPORTED_ABOVE_ZERO', 'REPORTED_ZERO')) AS link_outcome,
        reported_budget_pesos IS NOT NULL AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
    UNION ALL
    SELECT
        'OFFICIAL_FLOOD_LIST' AS integration_link,
        flood_list_match_status AS link_outcome,
        COALESCE(is_in_official_flood_list, FALSE) AS is_linked,
        region_key,
        reported_budget_pesos
    FROM projects
)

SELECT
    integration_link,
    link_outcome,
    is_linked,
    region_key,
    COUNT(*) AS project_count,
    COUNT_IF(reported_budget_pesos IS NOT NULL) AS projects_with_budget,
    SUM(reported_budget_pesos) AS budget
FROM links
GROUP BY ALL;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_ic_compare AS
WITH expected AS (
    SELECT
        analytics_validation_ic_expected.*,
        ROUND(100.0 * project_count / NULLIF(SUM(project_count) OVER (PARTITION BY integration_link), 0), 4)
            AS link_share,
        ROUND(100.0 * budget / NULLIF(SUM(budget) OVER (PARTITION BY integration_link), 0), 4) AS budget_share,
        ROUND(
            100.0 * project_count
            / NULLIF(SUM(project_count) OVER (PARTITION BY integration_link, region_key), 0),
            4
        ) AS region_share
    FROM analytics_validation_ic_expected
)

SELECT
    COALESCE(expected.integration_link, output.integration_link) AS integration_link,
    COALESCE(expected.link_outcome, output.link_outcome) AS link_outcome,
    COALESCE(expected.is_linked, output.is_linked) AS is_linked,
    COALESCE(expected.region_key, output.region_key) AS region_key,
    expected.project_count IS NULL AS extra_in_output,
    output.project_count IS NULL AS missing_from_output,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.projects_with_budget <=> expected.projects_with_budget)
    OR NOT (output.reported_budget_pesos <=> expected.budget) AS count_budget_break,
    NOT (output.pct_of_link_projects <=> expected.link_share)
    OR NOT (output.pct_of_link_budget <=> expected.budget_share)
    OR NOT (output.pct_of_region_projects <=> expected.region_share) AS share_break,
    output.project_count IS NOT NULL AND output.connected_source IS NULL AS source_label_missing,
    expected.project_count AS expected_projects,
    expected.budget AS expected_budget
FROM expected
FULL OUTER JOIN analytics_validation_ic_output AS output
    ON expected.integration_link = output.integration_link
    AND expected.link_outcome <=> output.link_outcome
    AND expected.is_linked <=> output.is_linked
    AND expected.region_key = output.region_key;

SELECT
    integration_link,
    is_linked,
    SUM(expected_projects) AS projects,
    SUM(expected_budget) AS reported_budget_pesos
FROM analytics_validation_ic_compare
GROUP BY ALL
ORDER BY integration_link, is_linked DESC;

## 10. Turn the metrics into checks

**What:** reduce every comparison to one metrics row per view, then turn each metrics row into check rows.

**STOP** checks protect the six-view contract, grain, null keys, row sets, every rebuilt measure, ranks, shares, project and budget reconciliation with Gold, population reconciliation, the matched and unmatched accounting of geography, flood, category, and status, percentage ranges, `NULL` handling, key `0` at both levels, the region and province hierarchy, BARMM, and lineage. A failed STOP check is `FAIL`.

**FLAG** checks keep known source limits visible with their counts, such as projects on key `0`, projects without a year or a budget, regions outside the equity comparison, flood points that are not in one level, projects that need review, projects without a map point, and the unlinked projects of every integration link. A FLAG is not a failure. Zero findings are `PASS`.

Every check names the columns it tested in `details`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_layer_metrics AS
SELECT
    COUNT(*) AS layer_views,
    CARDINALITY(expected_views) AS expected_view_count,
    CARDINALITY(expected_views) - COUNT_IF(is_expected) AS missing_expected_views,
    COUNT_IF(NOT is_expected) AS unexpected_views,
    COUNT_IF(is_retired) AS retired_views_found,
    COUNT_IF(table_name = 'analytics_region_project_progress') AS region_progress_views,
    COALESCE(CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CASE WHEN NOT is_expected THEN table_name END))), '')
        AS unexpected_detail
FROM analytics_validation_layer;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_inv_metrics AS
WITH gold_metrics AS (
    SELECT
        COUNT(*) AS gold_rows,
        COUNT(DISTINCT project_key) AS gold_projects,
        COALESCE(SUM(reported_budget_pesos), 0) AS gold_budget,
        COUNT_IF(region_key = 0) AS key0_projects,
        COALESCE(SUM(IF(region_key = 0, reported_budget_pesos, NULL)), 0) AS key0_budget,
        COUNT_IF(region_key <> 0 AND NOT region_found) AS orphan_projects,
        COALESCE(SUM(IF(region_key <> 0 AND NOT region_found, reported_budget_pesos, NULL)), 0) AS orphan_budget,
        COUNT_IF(reporting_area_key = 0) AS area_key0_projects,
        COALESCE(SUM(IF(reporting_area_key = 0, reported_budget_pesos, NULL)), 0) AS area_key0_budget,
        COUNT_IF(reporting_area_key <> 0 AND NOT area_found) AS area_orphan_projects,
        COALESCE(SUM(IF(reporting_area_key <> 0 AND NOT area_found, reported_budget_pesos, NULL)), 0)
            AS area_orphan_budget,
        COUNT_IF(region_key <> 0 AND region_found AND infra_year IS NULL) AS null_year_projects,
        COALESCE(SUM(IF(region_key <> 0 AND region_found AND infra_year IS NULL, reported_budget_pesos, NULL)), 0)
            AS null_year_budget,
        COUNT_IF(region_key <> 0 AND region_found AND reported_budget_pesos IS NULL) AS missing_budget_projects,
        COUNT_IF(region_key <> 0 AND region_found) AS included_projects,
        COUNT_IF(region_key <> 0 AND reporting_area_key = 0) AS region_without_area_projects
    FROM analytics_validation_current
),

key0_counts AS (
    SELECT
        region_match_status,
        COUNT(*) AS projects
    FROM analytics_validation_current
    WHERE region_key = 0
    GROUP BY ALL
),

key0_status AS (
    SELECT CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CONCAT(COALESCE(region_match_status, 'NULL'), '=', projects))))
        AS key0_detail
    FROM key0_counts
),

area_key0_counts AS (
    SELECT
        reporting_area_match_status,
        COUNT(*) AS projects
    FROM analytics_validation_current
    WHERE reporting_area_key = 0
    GROUP BY ALL
),

area_key0_status AS (
    SELECT CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CONCAT(COALESCE(reporting_area_match_status, 'NULL'), '=', projects))))
        AS area_key0_detail
    FROM area_key0_counts
),

output_metrics AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT NAMED_STRUCT('level', geography_level, 'key', geography_key, 'year', infra_year)) AS output_keys,
        COUNT_IF(geography_level IS NULL OR geography_key IS NULL OR cohort_row_type IS NULL) AS null_key_rows,
        COUNT(DISTINCT geography_level) AS output_levels,
        COUNT_IF(NOT geography_level IN ('REGION', 'REPORTING_AREA')) AS unexpected_level_rows,
        COUNT_IF(geography_key = 0) AS output_key0_rows,
        COALESCE(SUM(IF(geography_level = 'REGION', project_count, 0)), 0) AS output_region_projects,
        COALESCE(SUM(IF(geography_level = 'REGION', total_reported_budget_pesos, NULL)), 0) AS output_region_budget,
        COALESCE(SUM(IF(geography_level = 'REPORTING_AREA', project_count, 0)), 0) AS output_area_projects,
        COALESCE(SUM(IF(geography_level = 'REPORTING_AREA', total_reported_budget_pesos, NULL)), 0) AS output_area_budget,
        COUNT_IF(
            cohort_budget_share_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR cohort_cumulative_budget_share_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR geography_budget_share_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR geography_cumulative_budget_share_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
        ) AS out_of_range_rows,
        COUNT_IF(source_coverage_status <> reported_coverage_status AND (
            cohort_budget_rank IS NOT NULL OR cohort_project_count_rank IS NOT NULL
            OR geography_budget_rank IS NOT NULL OR geography_project_count_rank IS NOT NULL
        )) AS unreported_ranked_rows
    FROM analytics_validation_inv_output
),

member_rows AS (
    SELECT
        geography_level,
        geography_key,
        COUNT(DISTINCT NAMED_STRUCT(
            'status', source_coverage_status, 'projects', geography_project_count,
            'budget', geography_budget_pesos, 'share', geography_budget_share_pct,
            'rank', geography_budget_rank, 'count_rank', geography_project_count_rank,
            'cumulative', geography_cumulative_budget_share_pct, 'name', geography_name,
            'parent', parent_region_key, 'boundary', boundary_status
        )) AS member_variants,
        MAX(geography_budget_share_pct) AS member_share
    FROM analytics_validation_inv_output
    GROUP BY ALL
),

level_rows AS (
    SELECT
        geography_level,
        ABS(COALESCE(SUM(member_share), 100) - 100) AS level_share_gap
    FROM member_rows
    GROUP BY ALL
),

member_metrics AS (
    SELECT
        (SELECT COUNT_IF(member_variants > 1) FROM member_rows) AS inconsistent_members,
        COUNT_IF(level_share_gap > share_total_tolerance_pct) AS member_share_breaks
    FROM level_rows
),

cohort_totals AS (
    SELECT
        geography_level,
        infra_year,
        SUM(cohort_budget_share_pct) AS cohort_share_total
    FROM analytics_validation_inv_output
    WHERE cohort_row_type <> 'NO_MAPPED_PROJECT'
    GROUP BY ALL
    HAVING SUM(total_reported_budget_pesos) > 0
),

cohort_metrics AS (
    SELECT COUNT_IF(ABS(cohort_share_total - 100) > share_total_tolerance_pct) AS cohort_share_breaks
    FROM cohort_totals
),

compare_metrics AS (
    SELECT
        COUNT(*) AS compared_rows,
        COUNT_IF(extra_in_output OR missing_from_output) AS row_set_breaks,
        COUNT_IF(row_type_break) AS row_type_breaks,
        COUNT_IF(member_break) AS member_breaks,
        COUNT_IF(count_budget_break) AS count_budget_breaks,
        COUNT_IF(average_break) AS average_breaks,
        COUNT_IF(status_break) AS status_breaks,
        COUNT_IF(cohort_share_break) AS cohort_share_value_breaks,
        COUNT_IF(cohort_rank_break OR cohort_cumulative_break) AS cohort_rank_breaks,
        COUNT_IF(member_total_break) AS member_total_breaks,
        COUNT_IF(member_rank_break) AS member_rank_breaks,
        COUNT_IF(budget_without_projects) AS budget_without_project_rows,
        COUNT_IF(is_barmm AND source_coverage_status = reported_coverage_status) AS barmm_ranked_rows,
        COUNT(DISTINCT IF(source_coverage_status <> reported_coverage_status,
            NAMED_STRUCT('level', geography_level, 'key', geography_key), NULL)) AS unreported_members,
        COUNT(DISTINCT NAMED_STRUCT('level', geography_level, 'key', geography_key)) AS compared_members,
        COUNT(DISTINCT IF(geography_level = 'REGION', geography_key, NULL)) AS compared_regions,
        COUNT(DISTINCT IF(geography_level = 'REPORTING_AREA', geography_key, NULL)) AS compared_areas
    FROM analytics_validation_inv_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS lineage_breaks,
        COUNT_IF(NOT output.snapshot_date <=> context.snapshot_date) AS snapshot_date_breaks
    FROM analytics_validation_inv_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM gold_metrics
CROSS JOIN key0_status
CROSS JOIN area_key0_status
CROSS JOIN output_metrics
CROSS JOIN member_metrics
CROSS JOIN cohort_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_eq_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT reporting_area_key) AS output_areas,
        COUNT_IF(reporting_area_key IS NULL) AS null_key_rows,
        COUNT_IF(reporting_area_key = 0) AS output_key0_rows,
        COALESCE(SUM(population_count), 0) AS output_population,
        COALESCE(SUM(investment_share_pct), 0) AS investment_share_total,
        COALESCE(SUM(comparable_population_share_pct), 0) AS comparable_population_share_total,
        COALESCE(SUM(population_share_pct), 0) AS population_share_total,
        COUNT_IF(investment_share_pct IS NOT NULL) AS comparable_areas,
        COUNT_IF(
            population_share_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR comparable_population_share_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR investment_share_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
        ) AS out_of_range_rows
    FROM analytics_validation_eq_output
),

population_metrics AS (
    SELECT
        COALESCE(SUM(population_count), 0) AS gold_population,
        COUNT(*) AS gold_population_areas
    FROM analytics_validation_area_population
),

compare_metrics AS (
    SELECT
        COUNT(*) AS compared_areas,
        COUNT_IF(area_set_break) AS area_set_breaks,
        COUNT_IF(area_total_break) AS area_total_breaks,
        COUNT_IF(population_break) AS population_breaks,
        COUNT_IF(comparison_status_break) AS comparison_status_breaks,
        COUNT_IF(share_break) AS share_breaks,
        COUNT_IF(gap_break) AS gap_breaks,
        COUNT_IF(per_capita_break) AS per_capita_breaks,
        COUNT_IF(not_comparable_ratio_break) AS not_comparable_ratio_breaks,
        COUNT_IF(missing_population) AS areas_without_population,
        COUNT_IF(comparison_status <> 'COMPARABLE') AS excluded_areas,
        COALESCE(CONCAT_WS('; ', SORT_ARRAY(COLLECT_LIST(IF(comparison_status <> 'COMPARABLE',
            CONCAT(reporting_area_name, ' ', comparison_status), NULL)))), '') AS excluded_detail
    FROM analytics_validation_eq_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS lineage_breaks,
        COUNT_IF(output.population_count IS NOT NULL AND output.population_source_snapshot_id IS NULL)
            AS population_lineage_breaks
    FROM analytics_validation_eq_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN population_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_fa_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT NAMED_STRUCT('region', region_key, 'class', flood_point_class)) AS output_keys,
        COUNT_IF(region_key IS NULL OR flood_point_class IS NULL) AS null_key_rows,
        COUNT_IF(region_key = 0) AS output_key0_rows,
        COALESCE(SUM(project_count), 0) AS output_projects,
        COALESCE(SUM(total_reported_budget_pesos), 0) AS output_budget,
        COALESCE(SUM(flood_control_project_count), 0) AS output_flood_control_projects,
        COUNT_IF(
            share_of_region_flood_control_projects_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR share_of_region_flood_control_budget_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR share_of_region_area_pct NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
        ) AS out_of_range_rows
    FROM analytics_validation_fa_output
),

region_shares AS (
    SELECT
        region_key,
        SUM(flood_control_project_count) AS region_flood_projects,
        SUM(flood_control_budget_pesos) AS region_flood_budget,
        SUM(share_of_region_flood_control_projects_pct) AS region_project_share_total,
        SUM(share_of_region_flood_control_budget_pct) AS region_budget_share_total
    FROM analytics_validation_fa_output
    GROUP BY ALL
),

share_metrics AS (
    SELECT
        COUNT_IF(
            region_flood_projects > 0
            AND ABS(region_project_share_total - 100) > share_total_tolerance_pct
        )
        + COUNT_IF(
            region_flood_budget > 0
            AND ABS(region_budget_share_total - 100) > share_total_tolerance_pct
        ) AS share_total_breaks
    FROM region_shares
),

class_projects AS (
    SELECT
        region_key,
        SUM(project_count) AS class_projects
    FROM analytics_validation_fa_output
    GROUP BY ALL
),

inv_regions AS (
    -- one row per region. the member columns repeat on every cohort row of view 1
    SELECT DISTINCT
        geography_key AS region_key,
        IF(source_coverage_status = no_project_coverage_status, NULL, geography_project_count) AS inv_projects
    FROM analytics_validation_inv_output
    WHERE geography_level = 'REGION'
),

region_classes AS (
    -- each region's classes add up to its project count in view 1
    SELECT
        class_projects.region_key,
        class_projects.class_projects,
        inv_regions.inv_projects
    FROM class_projects
    LEFT JOIN inv_regions
        ON class_projects.region_key = inv_regions.region_key
),

region_metrics AS (
    SELECT COUNT_IF(NOT class_projects <=> inv_projects) AS region_total_breaks
    FROM region_classes
),

class_list AS (
    SELECT COLLECT_LIST(flood_point_class) AS published_classes
    FROM analytics_validation_fa_classes
),

gold_metrics AS (
    SELECT
        COUNT_IF(region_key <> 0 AND region_found) AS included_projects,
        COALESCE(SUM(IF(region_key <> 0 AND region_found, reported_budget_pesos, NULL)), 0) AS included_budget,
        COUNT_IF(region_key <> 0 AND region_found AND NOT ARRAY_CONTAINS(
            class_list.published_classes, COALESCE(expected_flood_point_class, 'NULL')
        )) AS unpublished_class_projects,
        COUNT_IF(region_key <> 0 AND region_found AND is_dpwh_flood_related) AS flood_control_projects,
        COUNT_IF(region_key <> 0 AND region_found AND is_dpwh_flood_related
            AND expected_flood_point_class IN ('AMBIGUOUS', 'OUTSIDE_MAPPED_ZONES', 'NO_USABLE_POINT', 'INVALID_SOURCE'))
            AS flood_control_without_level,
        COUNT_IF(region_key <> 0 AND region_found AND expected_flood_point_class = 'AMBIGUOUS') AS ambiguous_projects,
        COALESCE(CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(IF(region_key <> 0 AND region_found AND is_dpwh_flood_related
            AND expected_flood_point_class IN ('AMBIGUOUS', 'OUTSIDE_MAPPED_ZONES', 'NO_USABLE_POINT', 'INVALID_SOURCE'),
            expected_flood_point_class, NULL)))), '') AS flood_control_without_level_detail
    FROM analytics_validation_current
    CROSS JOIN class_list
),

compare_metrics AS (
    SELECT
        COUNT(*) AS compared_rows,
        COUNT_IF(extra_in_output OR missing_from_output) AS row_set_breaks,
        COUNT_IF(class_break) AS class_breaks,
        COUNT_IF(count_budget_break) AS count_budget_breaks,
        COUNT_IF(flood_measure_break) AS flood_measure_breaks,
        COUNT_IF(other_measure_break) AS other_measure_breaks,
        COUNT_IF(partition_break AND NOT is_unknown_region) AS partition_breaks,
        COUNT_IF(area_break) AS area_breaks,
        COUNT_IF(area_on_class_break) AS area_on_class_breaks,
        COUNT_IF(unknown_region_break) AS unknown_region_breaks,
        COUNT(DISTINCT IF(exposure_area_status <> additive_exposure_status, region_key, NULL)) AS non_additive_regions,
        COALESCE(CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(IF(exposure_area_status <> additive_exposure_status,
            CONCAT(CAST(region_key AS STRING), '=', exposure_area_status), NULL)))), '') AS non_additive_detail,
        COUNT(DISTINCT region_key) AS compared_regions
    FROM analytics_validation_fa_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS lineage_breaks,
        COUNT_IF(NOT output.mgb_source_snapshot_id <=> context.mgb_source_snapshot_id) AS mgb_lineage_breaks
    FROM analytics_validation_fa_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN share_metrics
CROSS JOIN region_metrics
CROSS JOIN gold_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_im_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT project_key) AS output_keys,
        COUNT_IF(project_key IS NULL) AS null_key_rows,
        COALESCE(SUM(reported_budget_pesos), 0) AS output_budget,
        COUNT_IF(physical_progress_pct NOT BETWEEN 0 AND 100) AS out_of_range_rows
    FROM analytics_validation_im_output
),

gold_metrics AS (
    SELECT
        COUNT(*) AS gold_rows,
        COUNT(DISTINCT project_key) AS gold_projects,
        COALESCE(SUM(reported_budget_pesos), 0) AS gold_budget,
        COUNT_IF(NOT project_found OR NOT region_found OR NOT status_found) AS missing_dimension_rows,
        COUNT_IF(status_group IS NULL) AS unresolved_status_projects,
        COALESCE(CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(IF(status_group IS NULL, expected_status_mapping_status, NULL)))),
            '') AS unresolved_status_detail,
        COUNT_IF(long_running_flag IS NULL) AS long_running_unknown
    FROM analytics_validation_current
),

compare_metrics AS (
    SELECT
        COUNT(*) AS compared_projects,
        COUNT_IF(extra_in_output OR missing_from_output) AS project_set_breaks,
        COUNT_IF(attribute_break) AS attribute_breaks,
        COUNT_IF(status_break) AS status_breaks,
        COUNT_IF(date_break) AS date_breaks,
        COUNT_IF(measure_break) AS measure_breaks,
        COUNT_IF(indicator_break) AS indicator_breaks,
        COUNT_IF(procurement_raised) AS procurement_raised_rows,
        COUNT_IF(unpublished_break) AS unpublished_breaks,
        COUNT_IF(progress_flag_disagrees OR flag_without_published_progress) AS progress_flag_mismatches,
        COUNT_IF(needs_review) AS review_projects,
        COALESCE(SUM(IF(needs_review, output_budget, NULL)), 0) AS review_budget,
        COUNT_IF(review_active_zero_progress) AS review_zero,
        COUNT_IF(review_active_long_running) AS review_long,
        COUNT_IF(review_active_progress_not_published) AS review_not_published,
        COUNT_IF(review_inactive_status) AS review_inactive,
        COUNT_IF(review_unresolved_status) AS review_unresolved
    FROM analytics_validation_im_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS lineage_breaks,
        COUNT_IF(NOT output.snapshot_date <=> context.snapshot_date) AS snapshot_date_breaks
    FROM analytics_validation_im_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN gold_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_net_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT project_key) AS output_keys,
        COUNT_IF(project_key IS NULL) AS null_key_rows,
        COALESCE(SUM(reported_budget_pesos), 0) AS output_budget
    FROM analytics_validation_net_output
),

gold_metrics AS (
    SELECT
        COUNT(*) AS gold_rows,
        COALESCE(SUM(reported_budget_pesos), 0) AS gold_budget,
        COALESCE(CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(IF(coordinate_status <=> 'VALID_PAIR', NULL,
            COALESCE(coordinate_status, 'NULL'))))), '') AS not_mappable_detail
    FROM analytics_validation_current
),

compare_metrics AS (
    SELECT
        COUNT(*) AS compared_projects,
        COUNT_IF(extra_in_output OR missing_from_output) AS project_set_breaks,
        COUNT_IF(attribute_break) AS attribute_breaks,
        COUNT_IF(point_break) AS point_breaks,
        COUNT_IF(grid_break) AS grid_breaks,
        COUNT_IF(flood_break) AS flood_breaks,
        COUNT_IF(population_break OR key0_population_break) AS population_breaks,
        COUNT_IF(hierarchy_break) AS hierarchy_breaks,
        COUNT_IF(hierarchy_conflict) AS hierarchy_conflicts,
        COUNT_IF(NOT is_mappable) AS not_mappable_projects,
        COUNT_IF(mappable_without_region) AS mappable_without_region
    FROM analytics_validation_net_compare
),

lineage_metrics AS (
    SELECT COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS lineage_breaks
    FROM analytics_validation_net_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN gold_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_ic_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT NAMED_STRUCT(
            'link', integration_link, 'outcome', link_outcome, 'linked', is_linked, 'region', region_key
        )) AS output_keys,
        COUNT_IF(integration_link IS NULL OR region_key IS NULL OR is_linked IS NULL) AS null_key_rows,
        COUNT(DISTINCT integration_link) AS output_links,
        COUNT_IF(NOT ARRAY_CONTAINS(expected_links, integration_link)) AS unexpected_link_rows,
        COUNT_IF(
            pct_of_link_projects NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR pct_of_link_budget NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
            OR pct_of_region_projects NOT BETWEEN 0 AND 100 + share_total_tolerance_pct
        ) AS out_of_range_rows,
        COUNT(DISTINCT NAMED_STRUCT('link', integration_link, 'outcome', link_outcome, 'region', region_key))
            AS output_outcome_keys
    FROM analytics_validation_ic_output
),

gold_metrics AS (
    SELECT
        COUNT(*) AS gold_projects,
        COALESCE(SUM(reported_budget_pesos), 0) AS gold_budget,
        COUNT_IF(region_key <> 0) AS gold_geography_linked,
        COUNT_IF(reporting_area_key <> 0) AS gold_area_linked
    FROM analytics_validation_current
),

link_sums AS (
    SELECT
        integration_link,
        SUM(project_count) AS link_projects,
        COALESCE(SUM(reported_budget_pesos), 0) AS link_budget,
        SUM(pct_of_link_projects) AS link_share_total,
        COALESCE(SUM(pct_of_link_budget), 0) AS link_budget_share_total,
        SUM(IF(is_linked, project_count, 0)) AS linked_projects
    FROM analytics_validation_ic_output
    GROUP BY ALL
),

link_totals AS (
    SELECT
        COUNT_IF(NOT (link_sums.link_projects <=> gold.gold_projects) OR NOT (link_sums.link_budget <=> gold.gold_budget))
            AS link_total_breaks,
        COUNT_IF(ABS(link_sums.link_share_total - 100) > share_total_tolerance_pct
            OR (gold.gold_budget > 0 AND ABS(link_sums.link_budget_share_total - 100) > share_total_tolerance_pct))
            AS link_share_breaks,
        MAX(IF(link_sums.integration_link = 'PSGC_REGION', link_sums.linked_projects, NULL)) AS geography_linked_projects,
        MAX(IF(link_sums.integration_link = 'PSGC_REPORTING_AREA', link_sums.linked_projects, NULL)) AS area_linked_projects
    FROM link_sums
    CROSS JOIN gold_metrics AS gold
),

unlinked AS (
    SELECT
        COALESCE(SUM(IF(integration_link = 'PSGC_REGION', expected_projects, 0)), 0) AS unlinked_region,
        COALESCE(SUM(IF(integration_link = 'PSGC_REPORTING_AREA', expected_projects, 0)), 0) AS unlinked_area,
        COALESCE(SUM(IF(integration_link = 'PSGC_REPORTING_AREA', expected_budget, NULL)), 0) AS unlinked_area_budget,
        COALESCE(SUM(IF(integration_link = 'REGION_PROVINCE_HIERARCHY' AND link_outcome = 'CONFLICT', expected_projects, 0)), 0)
            AS hierarchy_conflict_projects,
        COALESCE(SUM(IF(integration_link = 'POPULATION_VIA_REPORTING_AREA', expected_projects, 0)), 0)
            AS unlinked_area_population,
        COALESCE(SUM(IF(integration_link = 'COORDINATES', expected_projects, 0)), 0) AS unlinked_coordinates,
        COALESCE(SUM(IF(integration_link = 'POPULATION_VIA_REGION', expected_projects, 0)), 0) AS unlinked_population,
        COALESCE(SUM(IF(integration_link = 'FLOOD_SUSCEPTIBILITY', expected_projects, 0)), 0) AS unlinked_flood,
        COALESCE(SUM(IF(integration_link = 'PROJECT_CATEGORY', expected_projects, 0)), 0) AS unlinked_category,
        COALESCE(SUM(IF(integration_link = 'PROJECT_STATUS', expected_projects, 0)), 0) AS unlinked_status,
        COALESCE(SUM(IF(integration_link = 'REPORTED_BUDGET', expected_projects, 0)), 0) AS unlinked_budget,
        COALESCE(SUM(IF(integration_link = 'OFFICIAL_FLOOD_LIST', expected_projects, 0)), 0) AS unlinked_flood_list,
        COALESCE(SUM(IF(integration_link = 'PSGC_REGION', expected_budget, NULL)), 0) AS unlinked_region_budget,
        COALESCE(SUM(IF(integration_link = 'FLOOD_SUSCEPTIBILITY', expected_budget, NULL)), 0) AS unlinked_flood_budget,
        COALESCE(CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(IF(integration_link = 'FLOOD_SUSCEPTIBILITY', link_outcome, NULL)))),
            '') AS unlinked_flood_detail
    FROM analytics_validation_ic_compare
    WHERE NOT is_linked
),

compare_metrics AS (
    SELECT
        COUNT(*) AS compared_rows,
        COUNT_IF(extra_in_output OR missing_from_output) AS row_set_breaks,
        COUNT_IF(count_budget_break) AS count_budget_breaks,
        COUNT_IF(share_break) AS share_breaks,
        COUNT_IF(source_label_missing) AS source_label_breaks
    FROM analytics_validation_ic_compare
),

lineage_metrics AS (
    SELECT COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS lineage_breaks
    FROM analytics_validation_ic_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN gold_metrics
CROSS JOIN link_totals
CROSS JOIN unlinked
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_check_rows AS
WITH layer_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_layer', 'attribute', 'Completeness',
            'check_name', 'All six D-56 Analytics views exist',
            'failed_rows', CAST(m.missing_expected_views AS BIGINT), 'total_rows', CAST(m.expected_view_count AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: information_schema.views.table_name. Views in 05-analytics=', m.layer_views)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_layer', 'attribute', 'Validity',
            'check_name', 'No retired or unexpected view remains in 05-analytics',
            'failed_rows', CAST(m.unexpected_views AS BIGINT), 'total_rows', CAST(m.layer_views AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: information_schema.views.table_name. Retired views found=', m.retired_views_found, '. Unexpected: ', m.unexpected_detail)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_layer', 'attribute', 'Validity',
            'check_name', 'analytics_region_project_progress is not part of the Analytics layer',
            'failed_rows', CAST(m.region_progress_views AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', 'Columns: information_schema.views.table_name (D-55 withdrawn by D-56)'
        )
    ))
    FROM analytics_validation_layer_metrics AS m
),

inv_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Uniqueness',
            'check_name', 'One row per geography level, geography key, and infra_year cohort',
            'failed_rows', CAST(m.output_rows - m.output_keys AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: geography_level, geography_key, infra_year'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Uniqueness',
            'check_name', 'At most one current Gold row per project',
            'failed_rows', CAST(m.gold_rows - m.gold_projects AS BIGINT), 'total_rows', CAST(m.gold_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: fact_project_snapshot.project_key, is_current_snapshot. Protects every view from counting a project twice.'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Completeness',
            'check_name', 'No row has a NULL level, key, or row type, and only the two documented levels exist',
            'failed_rows', CAST(m.null_key_rows + m.unexpected_level_rows + ABS(m.output_levels - 2) AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: geography_level, geography_key, cohort_row_type. Levels in the view=', m.output_levels)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Validity',
            'check_name', 'Key 0 is never in the view at either level',
            'failed_rows', CAST(m.output_key0_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: geography_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Completeness',
            'check_name', 'Every official member and observed cohort of each level has its row and no extra row exists',
            'failed_rows', CAST(m.row_set_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: geography_level, geography_key, infra_year, cohort_row_type. Regions=', m.compared_regions, ', reporting areas=', m.compared_areas)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Validity',
            'check_name', 'Row type, member type, parent region, and boundary status match Gold',
            'failed_rows', CAST(m.row_type_breaks + m.member_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: cohort_row_type, geography_type, parent_region_key, boundary_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Accuracy',
            'check_name', 'Project count, projects with a budget, and budget match Gold and a NULL budget stays NULL',
            'failed_rows', CAST(m.count_budget_breaks + m.budget_without_project_rows AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, projects_with_budget, total_reported_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Accuracy',
            'check_name', 'Average budget per project matches Gold',
            'failed_rows', CAST(m.average_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: avg_budget_per_project_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Validity',
            'check_name', 'Coverage status matches Gold',
            'failed_rows', CAST(m.status_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_coverage_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Accuracy',
            'check_name', 'Cohort share, ranks, and running share match Gold',
            'failed_rows', CAST(m.cohort_share_value_breaks + m.cohort_rank_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: cohort_budget_share_pct, cohort_budget_rank, cohort_project_count_rank, cohort_cumulative_budget_share_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Accuracy',
            'check_name', 'Member totals, share, ranks, and running share match Gold',
            'failed_rows', CAST(m.member_total_breaks + m.member_rank_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: geography_project_count, geography_budget_pesos, geography_budget_share_pct, geography_budget_rank, geography_project_count_rank, geography_cumulative_budget_share_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Consistency',
            'check_name', 'Member-level columns are the same on every cohort row of a member',
            'failed_rows', CAST(m.inconsistent_members AS BIGINT), 'total_rows', CAST(m.compared_members AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_coverage_status, geography_name, parent_region_key, boundary_status, and the geography_ columns'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Consistency',
            'check_name', 'Cohort shares add up to 100 within each level and cohort and member shares add up to 100 within each level',
            'failed_rows', CAST(m.cohort_share_breaks + m.member_share_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: cohort_budget_share_pct, geography_budget_share_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Validity',
            'check_name', 'Shares and running shares are between 0 and 100',
            'failed_rows', CAST(m.out_of_range_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: cohort_budget_share_pct, cohort_cumulative_budget_share_pct, geography_budget_share_pct, geography_cumulative_budget_share_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Consistency',
            'check_name', 'Each level plus key 0 adds up to every current project and budget',
            'failed_rows', CAST(IF(m.output_region_projects + m.key0_projects + m.orphan_projects <> m.gold_projects OR m.output_region_budget + m.key0_budget + m.orphan_budget <> m.gold_budget, 1, 0) + IF(m.output_area_projects + m.area_key0_projects + m.area_orphan_projects <> m.gold_projects OR m.output_area_budget + m.area_key0_budget + m.area_orphan_budget <> m.gold_budget, 1, 0) AS BIGINT), 'total_rows', CAST(2 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: geography_level, project_count, total_reported_budget_pesos. Gold projects=', m.gold_projects, ', budget=', m.gold_budget, '. REGION view=', m.output_region_projects, ' + key 0=', m.key0_projects, ' + unknown keys=', m.orphan_projects, ', budget=', m.output_region_budget, ' + ', m.key0_budget, ' + ', m.orphan_budget, '. REPORTING_AREA view=', m.output_area_projects, ' + key 0=', m.area_key0_projects, ' + unknown keys=', m.area_orphan_projects, ', budget=', m.output_area_budget, ' + ', m.area_key0_budget, ' + ', m.area_orphan_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Validity',
            'check_name', 'Only REPORTED members are ranked and BARMM is never ranked',
            'failed_rows', CAST(m.unreported_ranked_rows + m.barmm_ranked_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_coverage_status, cohort_budget_rank, geography_budget_rank'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID and snapshot date are the current DPWH snapshot',
            'failed_rows', CAST(m.lineage_breaks + m.snapshot_date_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id, snapshot_date (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Completeness',
            'check_name', 'Current projects left out on region key 0',
            'failed_rows', CAST(m.key0_projects AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: geography_key, region_match_status. analytics_integration_coverage shows them. Reported budget=', m.key0_budget, '. By match status: ', COALESCE(m.key0_detail, 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Completeness',
            'check_name', 'Current projects left out on reporting area key 0',
            'failed_rows', CAST(m.area_key0_projects AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: geography_key, reporting_area_match_status (D-57). Reported budget=', m.area_key0_budget, '. Projects with an official region but no reporting area=', m.region_without_area_projects, '. By match status: ', COALESCE(m.area_key0_detail, 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Completeness',
            'check_name', 'Projects with no infrastructure year (infra_year NULL)',
            'failed_rows', CAST(m.null_year_projects AS BIGINT), 'total_rows', CAST(m.included_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: infra_year, cohort_row_type. They keep their own NO_INFRA_YEAR rows. Region-level reported budget=', m.null_year_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Completeness',
            'check_name', 'Projects in the view with no reported budget',
            'failed_rows', CAST(m.missing_budget_projects AS BIGINT), 'total_rows', CAST(m.included_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: projects_with_budget, total_reported_budget_pesos. Counted at region level. A missing budget stays NULL.'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_investment_distribution', 'attribute', 'Completeness',
            'check_name', 'Members with no or limited DPWH project coverage',
            'failed_rows', CAST(m.unreported_members AS BIGINT), 'total_rows', CAST(m.compared_members AS BIGINT),
            'action', 'flag',
            'details', 'Columns: geography_level, source_coverage_status, boundary_status. Not ranked. A reporting area without a valid boundary cannot receive projects by coordinates.'
        )
    ))
    FROM analytics_validation_inv_metrics AS m
),

eq_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Uniqueness',
            'check_name', 'One row per province reporting area',
            'failed_rows', CAST(m.output_rows - m.output_areas + m.null_key_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: reporting_area_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Validity',
            'check_name', 'Per-person ratios leave key 0 out',
            'failed_rows', CAST(m.output_key0_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: reporting_area_key, investment_per_capita_pesos (#61)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Completeness',
            'check_name', 'Every geographic reporting area has its row and no extra row exists',
            'failed_rows', CAST(m.area_set_breaks AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'stop',
            'details', 'Columns: reporting_area_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Consistency',
            'check_name', 'Coverage, parent region, project count, and budget match the Gold reporting-area totals',
            'failed_rows', CAST(m.area_total_breaks AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_coverage_status, region_key, project_count, total_reported_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Population, year, and source match the newest Gold primary reporting-area population',
            'failed_rows', CAST(m.population_breaks AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'stop',
            'details', 'Columns: population_count, population_reference_year, population_source_name'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Consistency',
            'check_name', 'View population adds up to the Gold primary reporting-area population',
            'failed_rows', CAST(IF(m.output_population = m.gold_population, 0, 1) AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: population_count. View=', m.output_population, ', Gold=', m.gold_population, ' over ', m.gold_population_areas, ' geographic reporting areas. Key 0 population is not in either.')
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Validity',
            'check_name', 'Comparison status matches Gold',
            'failed_rows', CAST(m.comparison_status_breaks AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'stop',
            'details', 'Columns: equity_comparison_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Population, comparable population, and investment shares match Gold',
            'failed_rows', CAST(m.share_breaks AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'stop',
            'details', 'Columns: population_share_pct, comparable_population_share_pct, investment_share_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Share gap and share ratio use the comparable population share',
            'failed_rows', CAST(m.gap_breaks AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'stop',
            'details', 'Columns: share_gap_pp, investment_to_population_share_ratio'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Per-capita values and rank match Gold',
            'failed_rows', CAST(m.per_capita_breaks AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'stop',
            'details', 'Columns: investment_per_capita_pesos, investment_per_capita_rank, comparable_set_per_capita_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Validity',
            'check_name', 'An area outside the comparable set shows no share, gap, ratio, or per-capita value',
            'failed_rows', CAST(m.not_comparable_ratio_breaks AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'stop',
            'details', 'Columns: investment_share_pct, comparable_population_share_pct, share_gap_pp, investment_to_population_share_ratio, investment_per_capita_pesos, investment_per_capita_rank'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Consistency',
            'check_name', 'Investment, comparable population, and population shares each add up to 100',
            'failed_rows', CAST(IF(m.comparable_areas > 0 AND ABS(m.investment_share_total - 100) > share_total_tolerance_pct, 1, 0) + IF(m.comparable_areas > 0 AND ABS(m.comparable_population_share_total - 100) > share_total_tolerance_pct, 1, 0) + IF(m.gold_population > 0 AND ABS(m.population_share_total - 100) > share_total_tolerance_pct, 1, 0) AS BIGINT), 'total_rows', CAST(3 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: investment_share_pct, comparable_population_share_pct, population_share_pct. Totals=', m.investment_share_total, ', ', m.comparable_population_share_total, ', ', m.population_share_total)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Validity',
            'check_name', 'Shares are between 0 and 100',
            'failed_rows', CAST(m.out_of_range_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: population_share_pct, comparable_population_share_pct, investment_share_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Auditability',
            'check_name', 'Snapshot IDs are the current DPWH snapshot and the population snapshot',
            'failed_rows', CAST(m.lineage_breaks + m.population_lineage_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id, population_source_snapshot_id (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Completeness',
            'check_name', 'Reporting areas outside the equity comparison',
            'failed_rows', CAST(m.excluded_areas AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: equity_comparison_status. ', m.excluded_detail)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Completeness',
            'check_name', 'Reporting areas without a primary population',
            'failed_rows', CAST(m.areas_without_population AS BIGINT), 'total_rows', CAST(m.compared_areas AS BIGINT),
            'action', 'flag',
            'details', 'Columns: population_count. Gold has no primary population value for these areas.'
        )
    ))
    FROM analytics_validation_eq_metrics AS m
),

fa_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Uniqueness',
            'check_name', 'One row per region and flood point class',
            'failed_rows', CAST(m.output_rows - m.output_keys AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, flood_point_class'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Completeness',
            'check_name', 'No row has a NULL region key or class, and key 0 is never in the view',
            'failed_rows', CAST(m.null_key_rows + m.output_key0_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, flood_point_class'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Completeness',
            'check_name', 'Every region has one row for every class and no extra row exists',
            'failed_rows', CAST(m.row_set_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: region_key, flood_point_class. Regions=', m.compared_regions)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Completeness',
            'check_name', 'Every current project outside key 0 lands in a published class',
            'failed_rows', CAST(m.unpublished_class_projects AS BIGINT), 'total_rows', CAST(m.included_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: fact_project_snapshot.flood_match_status, coordinate_status, flood_point_class'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Validity',
            'check_name', 'Class attributes and coverage status match Gold',
            'failed_rows', CAST(m.class_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: is_mgb_level, severity_rank, source_coverage_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Accuracy',
            'check_name', 'Project count and budget of each class match Gold and a NULL budget stays NULL',
            'failed_rows', CAST(m.count_budget_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, total_reported_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Accuracy',
            'check_name', 'Flood-control, strict, and official flood-list figures match Gold',
            'failed_rows', CAST(m.flood_measure_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_project_count, flood_control_budget_pesos, flood_control_single_sector_project_count, flood_control_single_sector_budget_pesos, official_flood_list_project_count, official_flood_list_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Consistency',
            'check_name', 'Flood-control, other, and uncategorized add up to the project count',
            'failed_rows', CAST(m.partition_breaks + m.other_measure_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_project_count, other_project_count, uncategorized_project_count, project_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Consistency',
            'check_name', 'Each region adds up to its project count in view 1, and the view adds up to Gold outside key 0',
            'failed_rows', CAST(m.region_total_breaks + IF(m.output_projects = m.included_projects AND m.output_budget = m.included_budget, 0, 1) AS BIGINT), 'total_rows', CAST(m.compared_regions + 1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_count, total_reported_budget_pesos. View projects=', m.output_projects, ', Gold outside key 0=', m.included_projects, '. View budget=', m.output_budget, ', Gold=', m.included_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Accuracy',
            'check_name', 'Level areas and exposure status match the newest Gold exposure run',
            'failed_rows', CAST(m.area_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: susceptible_area_sqkm, share_of_region_area_pct, exposure_area_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Validity',
            'check_name', 'Area appears only on MGB level rows',
            'failed_rows', CAST(m.area_on_class_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: is_mgb_level, susceptible_area_sqkm, share_of_region_area_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Validity',
            'check_name', 'A region with no mapped DPWH project keeps NULL project figures',
            'failed_rows', CAST(m.unknown_region_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_coverage_status, project_count, flood_control_project_count, total_reported_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Consistency',
            'check_name', 'Flood-control shares add up to 100 within each region',
            'failed_rows', CAST(m.share_total_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: share_of_region_flood_control_projects_pct, share_of_region_flood_control_budget_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Validity',
            'check_name', 'Shares are between 0 and 100',
            'failed_rows', CAST(m.out_of_range_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: share_of_region_flood_control_projects_pct, share_of_region_flood_control_budget_pct, share_of_region_area_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Auditability',
            'check_name', 'Snapshot IDs are the current DPWH and MGB snapshots',
            'failed_rows', CAST(m.lineage_breaks + m.mgb_lineage_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id, mgb_source_snapshot_id (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Completeness',
            'check_name', 'Flood-control projects not placed in one MGB level',
            'failed_rows', CAST(m.flood_control_without_level AS BIGINT), 'total_rows', CAST(m.flood_control_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: flood_point_class. Outside mapped zones is not low risk. Classes: ', m.flood_control_without_level_detail)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Completeness',
            'check_name', 'Projects whose point is in more than one MGB level (AMBIGUOUS)',
            'failed_rows', CAST(m.ambiguous_projects AS BIGINT), 'total_rows', CAST(m.included_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: flood_point_class. No level is picked (D-45).'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_alignment', 'attribute', 'Completeness',
            'check_name', 'Regions whose flood levels overlap or have no evaluated exposure',
            'failed_rows', CAST(m.non_additive_regions AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: exposure_area_status. Do not add level areas there. ', m.non_additive_detail)
        )
    ))
    FROM analytics_validation_fa_metrics AS m
),

im_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Uniqueness',
            'check_name', 'One row per current project',
            'failed_rows', CAST(m.output_rows - m.output_keys + m.null_key_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Completeness',
            'check_name', 'Every current Gold project has its row and no extra row exists',
            'failed_rows', CAST(m.project_set_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Integrity',
            'check_name', 'Every current project finds its dim_project, dim_region, and dim_project_status row',
            'failed_rows', CAST(m.missing_dimension_rows AS BIGINT), 'total_rows', CAST(m.gold_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_key, region_key, status_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Consistency',
            'check_name', 'Rows and budget add up to every current Gold project and budget',
            'failed_rows', CAST(IF(m.output_rows = m.gold_rows AND m.output_budget = m.gold_budget, 0, 1) AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_key, reported_budget_pesos. View rows=', m.output_rows, ', Gold=', m.gold_rows, '. View budget=', m.output_budget, ', Gold=', m.gold_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Accuracy',
            'check_name', 'Project, region, sector, office, and cohort match Gold',
            'failed_rows', CAST(m.attribute_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: contract_id, region_key, region_name, region_match_status, standardized_sector, infra_type_mapping_status, implementing_office, infra_year'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Accuracy',
            'check_name', 'Status, status reason, and For Procurement match Gold',
            'failed_rows', CAST(m.status_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_status, standardized_status, status_group, status_mapping_status, is_for_procurement'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Accuracy',
            'check_name', 'Start and completion dates match Gold and a missing date stays NULL',
            'failed_rows', CAST(m.date_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: start_date, source_completion_date'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Accuracy',
            'check_name', 'Progress, flags, and budget match Gold and unavailable values stay NULL',
            'failed_rows', CAST(m.measure_breaks + m.unpublished_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: physical_progress_pct, progress_reporting_status, zero_progress_flag, long_running_flag, reported_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Accuracy',
            'check_name', 'Review indicators, reason count, and needs_review follow the documented rules',
            'failed_rows', CAST(m.indicator_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: review_active_zero_progress, review_active_long_running, review_active_progress_not_published, review_inactive_status, review_unresolved_status, review_reason_count, needs_review'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Validity',
            'check_name', 'A For Procurement project never raises a progress indicator',
            'failed_rows', CAST(m.procurement_raised_rows AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: is_for_procurement, review_active_zero_progress, review_active_progress_not_published (D-46)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Validity',
            'check_name', 'Physical progress is between 0 and 100 or NULL',
            'failed_rows', CAST(m.out_of_range_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: physical_progress_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID and snapshot date are the current DPWH snapshot',
            'failed_rows', CAST(m.lineage_breaks + m.snapshot_date_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id, snapshot_date (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Completeness',
            'check_name', 'Projects with at least one review indicator',
            'failed_rows', CAST(m.review_projects AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: needs_review. A reason to verify, not a finding. Active zero progress=', m.review_zero, ', active long-running=', m.review_long, ', active progress not published=', m.review_not_published, ', inactive=', m.review_inactive, ', unresolved status=', m.review_unresolved, '. Reported budget=', m.review_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Completeness',
            'check_name', 'Projects with no mapped status group (status_group NULL)',
            'failed_rows', CAST(m.unresolved_status_projects AS BIGINT), 'total_rows', CAST(m.gold_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: status_group, status_mapping_status. Reasons: ', m.unresolved_status_detail)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Completeness',
            'check_name', 'Projects without a long-running flag',
            'failed_rows', CAST(m.long_running_unknown AS BIGINT), 'total_rows', CAST(m.gold_rows AS BIGINT),
            'action', 'flag',
            'details', 'Columns: long_running_flag. NULL when the start date is missing or the span cannot be measured (D-32, D-49).'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementation_monitoring', 'attribute', 'Consistency',
            'check_name', 'Zero-progress flag and published progress disagree',
            'failed_rows', CAST(m.progress_flag_mismatches AS BIGINT), 'total_rows', CAST(m.gold_rows AS BIGINT),
            'action', 'flag',
            'details', 'Columns: zero_progress_flag, physical_progress_pct. Both stay as Gold publishes them.'
        )
    ))
    FROM analytics_validation_im_metrics AS m
),

net_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Uniqueness',
            'check_name', 'One row per current project',
            'failed_rows', CAST(m.output_rows - m.output_keys + m.null_key_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Completeness',
            'check_name', 'Every current Gold project has its row and no extra row exists',
            'failed_rows', CAST(m.project_set_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Consistency',
            'check_name', 'Rows and budget add up to every current Gold project and budget',
            'failed_rows', CAST(IF(m.output_rows = m.gold_rows AND m.output_budget = m.gold_budget, 0, 1) AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_key, reported_budget_pesos. View rows=', m.output_rows, ', Gold=', m.gold_rows, '. View budget=', m.output_budget, ', Gold=', m.gold_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Accuracy',
            'check_name', 'Project, region, sector, cohort, status, and budget match Gold',
            'failed_rows', CAST(m.attribute_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: contract_id, region_key, region_name, region_match_status, standardized_sector, infra_type_mapping_status, is_dpwh_flood_related, is_in_official_flood_list, infra_year, status_group, status_mapping_status, reported_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Accuracy',
            'check_name', 'Points match Gold and only valid pairs are mappable',
            'failed_rows', CAST(m.point_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: latitude, longitude, coordinate_status, is_mappable'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Validity',
            'check_name', 'Each mappable point sits in its own display cell and other rows have none',
            'failed_rows', CAST(m.grid_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: map_grid_latitude, map_grid_longitude'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Accuracy',
            'check_name', 'Flood point class and severity rank match Gold',
            'failed_rows', CAST(m.flood_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_point_class, flood_severity_rank'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Accuracy',
            'check_name', 'Region and reporting-area population match Gold and key 0 has none',
            'failed_rows', CAST(m.population_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_population_count, population_reference_year, reporting_area_population_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Consistency',
            'check_name', 'Region and province hierarchy status follows the two Gold keys',
            'failed_rows', CAST(m.hierarchy_breaks AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, reporting_area_key, region_province_hierarchy_status (D-57)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.lineage_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Completeness',
            'check_name', 'Projects without a mappable point',
            'failed_rows', CAST(m.not_mappable_projects AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: is_mappable, coordinate_status. A map gap can be a project without a point.'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Completeness',
            'check_name', 'Mappable projects on region key 0',
            'failed_rows', CAST(m.mappable_without_region AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: is_mappable, region_key. The point is drawn but the project has no official region.'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_network', 'attribute', 'Consistency',
            'check_name', 'Projects whose region differs from the parent region of their reporting area',
            'failed_rows', CAST(m.hierarchy_conflicts AS BIGINT), 'total_rows', CAST(m.compared_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: region_province_hierarchy_status. Region comes from the reported name first and province from the map point (D-31, D-57). Both stay as Gold publishes them.'
        )
    ))
    FROM analytics_validation_net_metrics AS m
),

ic_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Uniqueness',
            'check_name', 'One row per link, outcome, and region key',
            'failed_rows', CAST(m.output_rows - m.output_keys + m.output_rows - m.output_outcome_keys AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: integration_link, link_outcome, is_linked, region_key. is_linked must follow the outcome.'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'No row has a NULL link, region key, or linked flag',
            'failed_rows', CAST(m.null_key_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: integration_link, region_key, is_linked'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Exactly the eleven documented links are present',
            'failed_rows', CAST(ABS(m.output_links - CARDINALITY(expected_links)) + m.unexpected_link_rows AS BIGINT), 'total_rows', CAST(CARDINALITY(expected_links) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: integration_link. Links in the view=', m.output_links)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Every Gold link outcome and region has its row and no extra row exists',
            'failed_rows', CAST(m.row_set_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: integration_link, link_outcome, is_linked, region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Accuracy',
            'check_name', 'Project count, projects with a budget, and budget match Gold and a NULL budget stays NULL',
            'failed_rows', CAST(m.count_budget_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, projects_with_budget, reported_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Consistency',
            'check_name', 'Matched and unmatched rows of every link add up to every current Gold project and budget',
            'failed_rows', CAST(m.link_total_breaks AS BIGINT), 'total_rows', CAST(m.output_links AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: integration_link, project_count, reported_budget_pesos. Gold projects=', m.gold_projects, ', Gold budget=', m.gold_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Consistency',
            'check_name', 'Geography-linked projects equal the Gold projects with an official region',
            'failed_rows', CAST(IF(m.geography_linked_projects <=> m.gold_geography_linked, 0, 1) AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: is_linked for PSGC_REGION. View=', COALESCE(CAST(m.geography_linked_projects AS STRING), 'NULL'), ', Gold=', m.gold_geography_linked)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Consistency',
            'check_name', 'Reporting-area-linked projects equal the Gold projects with a reporting area',
            'failed_rows', CAST(IF(m.area_linked_projects <=> m.gold_area_linked, 0, 1) AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: is_linked for PSGC_REPORTING_AREA. View=', COALESCE(CAST(m.area_linked_projects AS STRING), 'NULL'), ', Gold=', m.gold_area_linked)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Accuracy',
            'check_name', 'Shares match Gold',
            'failed_rows', CAST(m.share_breaks AS BIGINT), 'total_rows', CAST(m.compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_of_link_projects, pct_of_link_budget, pct_of_region_projects'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Consistency',
            'check_name', 'Shares of every link add up to 100',
            'failed_rows', CAST(m.link_share_breaks AS BIGINT), 'total_rows', CAST(m.output_links AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_of_link_projects, pct_of_link_budget'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Validity',
            'check_name', 'Shares are between 0 and 100 and every link names its source',
            'failed_rows', CAST(m.out_of_range_rows + m.source_label_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_of_link_projects, pct_of_link_budget, pct_of_region_projects, connected_source'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.lineage_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects not linked to an official PSGC region',
            'failed_rows', CAST(m.unlinked_region AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: integration_link PSGC_REGION, is_linked. Reported budget=', m.unlinked_region_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects not linked to a province reporting area',
            'failed_rows', CAST(m.unlinked_area AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: integration_link PSGC_REPORTING_AREA, link_outcome (D-57). Reported budget=', m.unlinked_area_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Consistency',
            'check_name', 'Projects whose region and reporting area disagree',
            'failed_rows', CAST(m.hierarchy_conflict_projects AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: integration_link REGION_PROVINCE_HIERARCHY, link_outcome CONFLICT. Both keys stay as Gold publishes them.'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects not linked to population through their reporting area',
            'failed_rows', CAST(m.unlinked_area_population AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: integration_link POPULATION_VIA_REPORTING_AREA, link_outcome'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects without a valid coordinate pair',
            'failed_rows', CAST(m.unlinked_coordinates AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: integration_link COORDINATES, is_linked'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects not linked to population through their region',
            'failed_rows', CAST(m.unlinked_population AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: integration_link POPULATION_VIA_REGION, is_linked'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects not placed in one MGB flood level',
            'failed_rows', CAST(m.unlinked_flood AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: integration_link FLOOD_SUSCEPTIBILITY, link_outcome. Not low risk. Reported budget=', m.unlinked_flood_budget, '. Outcomes: ', m.unlinked_flood_detail)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects without a governed category',
            'failed_rows', CAST(m.unlinked_category AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: integration_link PROJECT_CATEGORY, link_outcome'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects without a governed status group',
            'failed_rows', CAST(m.unlinked_status AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: integration_link PROJECT_STATUS, link_outcome'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects without a reported budget',
            'failed_rows', CAST(m.unlinked_budget AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: integration_link REPORTED_BUDGET, link_outcome. A missing budget is never counted as 0.'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_integration_coverage', 'attribute', 'Completeness',
            'check_name', 'Projects not in the official flood-control list',
            'failed_rows', CAST(m.unlinked_flood_list AS BIGINT), 'total_rows', CAST(m.gold_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: integration_link OFFICIAL_FLOOD_LIST, link_outcome. Membership, not a quality gap.'
        )
    ))
    FROM analytics_validation_ic_metrics AS m
)

-- each metrics row is turned into check rows, so no table is scanned again per check
SELECT * FROM layer_checks
UNION ALL
SELECT * FROM inv_checks
UNION ALL
SELECT * FROM eq_checks
UNION ALL
SELECT * FROM fa_checks
UNION ALL
SELECT * FROM im_checks
UNION ALL
SELECT * FROM net_checks
UNION ALL
SELECT * FROM ic_checks;

### Rerun stability

**What:** compare every check of this run with the evidence already saved under the same validation run ID, then add one STOP check for the result.

**Why:** the same Gold runs give the same validation run ID (D-52). A rerun on the same Gold data must give the same failed and total rows for every check, so the views and the rebuilds are deterministic. On the first run of a validation run ID, nothing is saved yet, so the check compares no rows and passes.

**Expected result:** `changed_checks` is 0.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.analytics_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Analytics validation evidence. One row per check per validation run.';

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_rerun_metrics AS
SELECT
    COUNT(saved.data_quality_check) AS compared_checks,
    COUNT_IF(saved.data_quality_check IS NOT NULL AND (
        NOT (saved.failed_rows <=> current_checks.failed_rows)
        OR NOT (saved.total_rows <=> current_checks.total_rows)
    )) AS changed_checks
FROM analytics_validation_check_rows AS current_checks
CROSS JOIN analytics_validation_context AS context
LEFT JOIN `04-validation`.analytics_dq_results AS saved
    ON saved.validation_run_id = context.validation_run_id
    AND saved.table_name = current_checks.table_name
    AND saved.data_quality_check = current_checks.check_name;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_checks AS
WITH rerun_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_layer', 'attribute', 'Consistency',
            'check_name', 'A rerun on the same Gold data gives the same check results',
            'failed_rows', CAST(m.changed_checks AS BIGINT), 'total_rows', CAST(m.compared_checks AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: failed_rows, total_rows of every saved check of this validation run. Checks compared=', m.compared_checks)
        )
    ))
    FROM analytics_validation_rerun_metrics AS m
),

checks AS (
    SELECT * FROM analytics_validation_check_rows
    UNION ALL
    SELECT * FROM rerun_checks
)

SELECT
    context.validation_run_id,
    COALESCE(
        CASE checks.table_name
            WHEN 'analytics_population_equity' THEN context.population_fact_run_id
            WHEN 'analytics_flood_alignment' THEN context.exposure_fact_run_id
            ELSE context.project_fact_run_id
        END,
        context.validation_run_id
    ) AS pipeline_run_id,
    'analytics' AS layer,
    checks.table_name,
    CASE checks.table_name
        WHEN 'analytics_flood_alignment' THEN context.mgb_source_snapshot_id
        ELSE context.project_snapshot_id
    END AS source_snapshot_id,
    validation_rule_version AS rule_version,
    checks.attribute AS data_quality_attribute,
    checks.check_name AS data_quality_check,
    checks.failed_rows,
    checks.total_rows,
    CASE WHEN checks.total_rows > 0
        THEN ROUND(100.0 * checks.failed_rows / checks.total_rows, 4) END AS percentage,
    CASE
        WHEN checks.failed_rows = 0 THEN 'PASS'
        WHEN checks.action = 'stop' THEN 'FAIL'
        ELSE 'FLAG'
    END AS status,
    checks.action,
    checks.details,
    CURRENT_TIMESTAMP() AS run_ts
FROM checks
CROSS JOIN analytics_validation_context AS context;

SELECT
    table_name,
    data_quality_attribute,
    data_quality_check,
    status,
    failed_rows,
    total_rows
FROM analytics_validation_checks
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END, table_name, data_quality_check;

## 11. Save the evidence before the gate

**What:** `MERGE` the checks into `04-validation.analytics_dq_results`, then show them, failures first.

**Why:** the saved rows stay reviewable even when the gate stops the run. The `MERGE` key is the validation run ID, the table, and the check name, so a rerun updates the same rows.

**Expected result:** one saved row per check, failures first.

In [0]:
%sql
MERGE INTO `04-validation`.analytics_dq_results AS target
USING analytics_validation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT
    table_name,
    status,
    action,
    failed_rows,
    total_rows,
    percentage,
    data_quality_check,
    details
FROM `04-validation`.analytics_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM analytics_validation_context)
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END, table_name, data_quality_check;

## 12. STOP gate

**What:** stop the run if any STOP check of this validation run failed, or if no evidence was saved.

**Expected result:** the `ASSERT_TRUE` returns `NULL`, and no error is raised.

In [0]:
%sql
SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking analytics checks failed or no evidence was saved'
    )
)
FROM `04-validation`.analytics_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM analytics_validation_context);

## Summary

This notebook validates the six-view Analytics contract of D-56, with the D-57 province reporting area, under rule version `analytics-validation-v4`. It proves that `05-analytics` holds exactly the six views, with no retired view and no `analytics_region_project_progress`. It rebuilds each view from Gold with separate SQL and checks grain, null keys, row sets, every measure, ranks, shares, project and budget reconciliation, population reconciliation, the matched and unmatched accounting of geography, flood, category, status, budget, and the official flood list, percentage ranges, `NULL` handling, key `0` at both levels, the region and province hierarchy, BARMM, and lineage. A rerun on the same Gold data must give the same results. Known source limits stay visible as FLAG rows with counts. Evidence from the earlier v1 and v2 contracts is retired and stays as history. See [analytics validation](../../docs/validation.md#analytics-validation).